
* 20251110
1. 抽取时必须加入元数据，参考dc 核心元数据，目标是最终的neo4j 中的对应节点（特别是chunk节点）都包含这些元数据。从而能够溯源，并支持支持同一个paper 内的分析。（done）
2. relation目前抽取时，没有保留原始文本信息，对照node具有WHU_HASORIGINALTEXT属性，缺少WHU_HASORIGINALTEXT属性，导致后期分析问题，故必须为relation补全这一属性。
3. scheam定义中缺少
   - Method-[supports]->data
   - Statement-[supports]->Claim
   特别是Statement-[supports]->Claim非常重要，必须补全（done）
4. 同时抽取中relation缺乏属性，包括weight，导致后期的路径，dependency等计算意义减弱，但如何定义weight，需要与实际的论文关系结合例如，整个kg中，activity与method的重复次数，在此使用多种方法来确定
   - 直接通过llm 来确定，其在抽取过程中就确定weight，但具有不稳定下
   - kg构建完成后，通过运算确定权重，例如，重复次数，路径长度等。这种方法具有可重复性，可解释的优点
   最终的weight，可以定义多个，包括llm_weihgt, math_weight, 根据需要进行选择
5. 在抽取得到的关系，节点中，需要表明其来源于的语义结构的部分，例如，来源于 Method, Abstract, Conclusion 等。（done）

# 模块0: 生成文档dc核心元数据Agent

In [1]:
"""
DC元数据自动提取Agent - 完整版
使用LangGraph让Agent自主选择工具：
- 英文论文 → search_crossref
- 中文论文 → search_chinese_metadata
"""

import os
import json
import yaml
import re
import requests
from typing import TypedDict, Annotated, Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode

# ==================== 配置 ====================
USER_EMAIL = "tomisacat@live.cn"
DEEPSEEK_API_KEY = "YOUR_DEEPSEEK_API_KEY"

# ==================== 工具1: CrossRef搜索（英文论文） ====================
@tool
def search_crossref(title: str, author: str = "") -> dict:
    """
    搜索英文论文元数据（使用CrossRef API）
    
    适用场景：英文学术论文
    
    Args:
        title: 论文英文标题（必需）
        author: 作者姓名（可选，提供可提高准确度）
    
    Returns:
        包含论文元数据的字典，包括标题、作者、DOI、期刊、日期等
    """
    try:
        url = "https://api.crossref.org/works"
        params = {
            "query": f"{title} {author}".strip(),
            "rows": 1,
            "select": "DOI,title,author,published,container-title,publisher,abstract"
        }
        headers = {
            "User-Agent": f"MetadataExtractor/1.0 (mailto:{USER_EMAIL})"
        }
        
        response = requests.get(url, params=params, headers=headers, timeout=15)
        response.raise_for_status()
        data = response.json()
        
        if not data['message']['items']:
            return {
                "success": False,
                "error": "No results found in CrossRef database",
                "source": "CrossRef"
            }
        
        item = data['message']['items'][0]
        
        # 提取作者
        authors = []
        if 'author' in item:
            authors = [
                f"{a.get('given', '')} {a.get('family', '')}".strip()
                for a in item['author'][:5]
            ]
        
        # 提取日期
        date = ""
        if 'published' in item and 'date-parts' in item['published']:
            parts = item['published']['date-parts'][0]
            if len(parts) >= 3:
                date = f"{parts[0]}-{parts[1]:02d}-{parts[2]:02d}"
            elif len(parts) >= 1:
                date = str(parts[0])
        
        # 提取摘要
        abstract = item.get('abstract', '')
        abstract = re.sub(r'<[^>]+>', '', abstract)
        
        return {
            "success": True,
            "dc_title": item.get('title', [''])[0],
            "dc_author": ", ".join(authors) if authors else "Unknown",
            "dc_creator": authors[0] if authors else "Unknown",
            "dc_publisher": item.get('container-title', ['Unknown'])[0],
            "dc_type": "Research Article",
            "dc_language": "en",
            "dcterms_issued": date or "Unknown",
            "dcterms_identifier": item.get('DOI', ''),
            "dcterms_abstract": abstract[:500] if abstract else "",
            "publisher_name": item.get('publisher', 'Unknown'),
            "source": "CrossRef"
        }
        
    except Exception as e:
        return {
            "success": False,
            "error": f"CrossRef search failed: {str(e)}",
            "source": "CrossRef"
        }


# ==================== 工具2: LLM提取（中文论文） ====================
@tool
def search_chinese_metadata(content: str, title: str = "") -> dict:
    """
    从中文论文文档中提取元数据（使用LLM深度分析）
    
    适用场景：中文学术论文、没有DOI的文档
    
    Args:
        content: 论文的完整Markdown内容（必需）
        title: 论文标题（可选，用于验证）
    
    Returns:
        包含论文元数据的字典，从文档内容中提取
    """
    try:
        llm = ChatOpenAI(
            model="deepseek-chat",
            api_key=DEEPSEEK_API_KEY,
            base_url="https://api.deepseek.com",
            temperature=0
        )
        
        prompt = f"""你是一个专业的学术元数据提取专家。请从以下中文论文文档中提取完整的Dublin Core元数据。

论文内容（前2000字符）：
```
{content[:2000]}
```

请提取以下信息（如果文档中没有，使用"Unknown"）：
1. dc_title: 论文完整标题（中文）
2. dc_author: 所有作者姓名，用逗号分隔
3. dc_creator: 第一作者姓名
4. dc_publisher: 期刊名称或出版社
5. dcterms_issued: 发表日期（格式：YYYY-MM-DD 或 YYYY）
6. dcterms_identifier: DOI或其他标识符（如果有）
7. dcterms_abstract: 摘要（不超过300字）
8. dc_subject: 关键词，用逗号分隔
9. institution: 作者机构
10. funding: 基金资助信息

以JSON格式返回（不要使用markdown代码块）：
{{
  "dc_title": "论文标题",
  "dc_author": "作者1, 作者2, 作者3",
  "dc_creator": "第一作者",
  "dc_publisher": "期刊名称",
  "dcterms_issued": "发表日期",
  "dcterms_identifier": "DOI或标识符",
  "dcterms_abstract": "摘要内容",
  "dc_subject": "关键词1, 关键词2, 关键词3",
  "institution": "作者机构",
  "funding": "基金资助"
}}
"""
        
        response = llm.invoke([HumanMessage(content=prompt)])
        text = response.content.strip()
        
        # 清理可能的代码块标记
        text = re.sub(r'```json\s*', '', text)
        text = re.sub(r'```\s*', '', text)
        
        metadata = json.loads(text)
        
        # 标准化输出格式
        return {
            "success": True,
            "dc_title": metadata.get('dc_title', title or 'Unknown'),
            "dc_author": metadata.get('dc_author', 'Unknown'),
            "dc_creator": metadata.get('dc_creator', 'Unknown'),
            "dc_publisher": metadata.get('dc_publisher', 'Unknown'),
            "dc_type": "Research Article",
            "dc_language": "zh",
            "dc_subject": metadata.get('dc_subject', ''),
            "dcterms_issued": metadata.get('dcterms_issued', 'Unknown'),
            "dcterms_identifier": metadata.get('dcterms_identifier', f"local:{title}"),
            "dcterms_abstract": metadata.get('dcterms_abstract', ''),
            "institution": metadata.get('institution', ''),
            "funding": metadata.get('funding', ''),
            "source": "LLM_Extraction"
        }
        
    except Exception as e:
        return {
            "success": False,
            "error": f"LLM extraction failed: {str(e)}",
            "source": "LLM_Extraction"
        }


# ==================== 辅助工具：保存元数据 ====================
@tool
def save_metadata(filepath: str, metadata: dict) -> dict:
    """
    保存元数据到JSON和YAML文件
    
    Args:
        filepath: 原始Markdown文件路径
        metadata: 要保存的元数据字典
    
    Returns:
        保存结果
    """
    try:
        base = os.path.splitext(filepath)[0]
        
        # 保存JSON
        json_path = f"{base}_metadata.json"
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)
        
        # 保存YAML
        yaml_path = f"{base}_metadata.yaml"
        with open(yaml_path, 'w', encoding='utf-8') as f:
            yaml.dump(metadata, f, allow_unicode=True, default_flow_style=False)
        
        return {
            "success": True,
            "json_path": json_path,
            "yaml_path": yaml_path
        }
    except Exception as e:
        return {
            "success": False,
            "error": f"Failed to save: {str(e)}"
        }


# ==================== 清空元数据文件 ====================
def clear_metadata_files(filepath: str) -> None:
    """清空已有的JSON和YAML元数据文件"""
    base = os.path.splitext(filepath)[0]
    json_path = f"{base}_metadata.json"
    yaml_path = f"{base}_metadata.yaml"
    
    cleared = []
    if os.path.exists(json_path):
        os.remove(json_path)
        cleared.append("JSON")
    if os.path.exists(yaml_path):
        os.remove(yaml_path)
        cleared.append("YAML")
    
    if cleared:
        print(f"   🗑️  已清空: {', '.join(cleared)}")


# ==================== State定义 ====================
class AgentState(TypedDict):
    messages: Annotated[list, "对话历史"]
    markdown_path: str
    content: str
    title: str
    author: str
    metadata: dict
    success: bool


# ==================== Agent节点 ====================
def read_file_node(state: AgentState) -> AgentState:
    """节点1: 读取文件"""
    try:
        with open(state['markdown_path'], 'r', encoding='utf-8') as f:
            state['content'] = f.read()
        print(f"✅ 读取: {os.path.basename(state['markdown_path'])}")
    except Exception as e:
        print(f"❌ 读取失败: {e}")
        state['content'] = ""
        state['success'] = False
    return state


def extract_basic_info(state: AgentState) -> AgentState:
    """节点2: 快速提取标题（用于Agent判断）"""
    
    if not state['content']:
        state['success'] = False
        return state
    
    # 简单规则提取标题
    lines = state['content'].split('\n')
    for line in lines[:30]:
        if line.strip().startswith('# ') and not line.strip().startswith('##'):
            state['title'] = line.lstrip('#').strip()
            break
    
    if not state['title']:
        state['title'] = os.path.basename(state['markdown_path']).replace('.md', '')
    
    print(f"📝 标题: {state['title'][:60]}...")
    return state


def agent_call_tools(state: AgentState) -> AgentState:
    """
    节点3: Agent决策并调用工具
    
    核心逻辑：
    1. LLM分析标题，判断是中文还是英文论文
    2. 自主选择调用 search_crossref 或 search_chinese_metadata
    3. 调用 save_metadata 保存结果
    """
    
    # 创建绑定工具的LLM
    llm = ChatOpenAI(
        model="deepseek-chat",
        api_key=DEEPSEEK_API_KEY,
        base_url="https://api.deepseek.com",
        temperature=0
    )
    
    # 绑定所有工具
    tools = [search_crossref, search_chinese_metadata, save_metadata]
    llm_with_tools = llm.bind_tools(tools)
    
    # 构建提示
    system_message = """你是一个学术元数据提取助手。你有以下工具：

1. search_crossref(title, author): 搜索英文论文（CrossRef数据库）
2. search_chinese_metadata(content, title): 从中文文档提取元数据（LLM分析）
3. save_metadata(filepath, metadata): 保存元数据到文件

任务流程：
1. 判断论文标题是中文还是英文
2. 选择合适的搜索工具：
   - 如果是英文标题 → 使用 search_crossref
   - 如果是中文标题 → 使用 search_chinese_metadata
3. 获得元数据后，使用 save_metadata 保存

注意：
- 中文论文必须传入完整的content参数
- 英文论文只需要title即可
- 如果一个工具失败，不要尝试另一个（除非你认为有必要）
"""
    
    user_message = f"""请帮我提取以下论文的元数据：

标题: {state['title']}
文件路径: {state['markdown_path']}

文档内容已经加载，你可以在调用工具时使用。

请分析标题语言，选择合适的工具提取元数据，然后保存到文件。
"""
    
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]
    
    print(f"🤖 Agent开始决策...")
    
    # Agent决策循环
    max_iterations = 5
    for i in range(max_iterations):
        # LLM决策
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # 检查是否有工具调用
        if not response.tool_calls:
            # 没有工具调用，结束
            print(f"   ✅ Agent完成决策")
            break
        
        # 执行工具调用
        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            
            print(f"   🔧 调用工具: {tool_name}")
            
            # 执行工具
            if tool_name == 'search_crossref':
                result = search_crossref.invoke(tool_args)
                if result['success']:
                    state['metadata'] = result
                    state['success'] = True
                    print(f"      ✅ CrossRef找到: {result['dc_title'][:50]}...")
                    print(f"      📄 DOI: {result['dcterms_identifier']}")
                else:
                    print(f"      ❌ CrossRef未找到: {result.get('error')}")
            
            elif tool_name == 'search_chinese_metadata':
                # 注入content参数
                tool_args['content'] = state['content']
                result = search_chinese_metadata.invoke(tool_args)
                if result['success']:
                    state['metadata'] = result
                    state['success'] = True
                    print(f"      ✅ LLM提取: {result['dc_title'][:50]}...")
                    print(f"      📚 作者: {result['dc_author']}")
                else:
                    print(f"      ❌ LLM提取失败: {result.get('error')}")
            
            elif tool_name == 'save_metadata':
                # 注入实际参数
                tool_args['filepath'] = state['markdown_path']
                tool_args['metadata'] = state['metadata']
                result = save_metadata.invoke(tool_args)
                if result['success']:
                    print(f"      💾 已保存: {os.path.basename(result['json_path'])}")
                else:
                    print(f"      ❌ 保存失败: {result.get('error')}")
            
            # 将工具结果添加到消息
            messages.append({
                "role": "tool",
                "content": json.dumps(result, ensure_ascii=False),
                "tool_call_id": tool_call['id']
            })
    
    # 如果没有成功，使用默认值
    if not state.get('success'):
        print(f"   ⚠️ 使用默认值")
        state['metadata'] = {
            "dc_title": state.get('title', 'Unknown'),
            "dc_author": "Unknown",
            "dc_creator": "Unknown",
            "dc_publisher": "Unknown",
            "dc_type": "Research Article",
            "dc_language": "unknown",
            "dcterms_issued": "Unknown",
            "dcterms_identifier": f"local:{os.path.basename(state['markdown_path'])}",
            "source": "default"
        }
        # 保存默认值
        save_metadata.invoke({
            "filepath": state['markdown_path'],
            "metadata": state['metadata']
        })
    
    return state


# ==================== 构建Graph ====================
def create_agent():
    """创建LangGraph工作流"""
    workflow = StateGraph(AgentState)
    
    # 添加节点
    workflow.add_node("read", read_file_node)
    workflow.add_node("extract", extract_basic_info)
    workflow.add_node("agent", agent_call_tools)
    
    # 定义流程
    workflow.set_entry_point("read")
    workflow.add_edge("read", "extract")
    workflow.add_edge("extract", "agent")
    workflow.add_edge("agent", END)
    
    return workflow.compile()


# ==================== 批量处理 ====================
def process_directory(directory: str, clear_existing: bool = True):
    """批量处理目录中的Markdown文件"""
    
    agent = create_agent()
    files = [f for f in os.listdir(directory) if f.endswith('.md')]
    
    print(f"\n{'='*60}")
    print(f"🚀 开始处理 {len(files)} 个文件")
    if clear_existing:
        print(f"🗑️  模式: 清空并重新生成元数据")
    print(f"{'='*60}\n")
    
    stats = {"success": 0, "failed": 0, "cleared": 0}
    
    for filename in files:
        filepath = os.path.join(directory, filename)
        json_path = os.path.splitext(filepath)[0] + '_metadata.json'
        yaml_path = os.path.splitext(filepath)[0] + '_metadata.yaml'
        
        print(f"📄 {filename}")
        print("-" * 60)
        
        # 检查并清空已有文件
        has_metadata = os.path.exists(json_path) or os.path.exists(yaml_path)
        if has_metadata and clear_existing:
            clear_metadata_files(filepath)
            stats["cleared"] += 1
        
        # 初始化状态
        initial = {
            "messages": [],
            "markdown_path": filepath,
            "content": "",
            "title": "",
            "author": "",
            "metadata": {},
            "success": False
        }
        
        try:
            final = agent.invoke(initial)
            if final.get('success'):
                stats["success"] += 1
                print("✅ 完成\n")
            else:
                stats["failed"] += 1
                print("⚠️ 完成（使用默认值）\n")
        except Exception as e:
            stats["failed"] += 1
            print(f"❌ 错误: {e}\n")
            import traceback
            traceback.print_exc()
    
    print(f"{'='*60}")
    print(f"📊 处理结果:")
    print(f"   成功: {stats['success']}")
    print(f"   失败: {stats['failed']}")
    if clear_existing:
        print(f"   已清空: {stats['cleared']}")
    print(f"{'='*60}\n")


# ==================== 主程序 手动运行====================
#if __name__ == "__main__":
    #process_directory(
    #    directory=r".\data\markdown\forTest",
    #    clear_existing=True)
  

# 模块1: 文档解析与元数据提取

In [2]:
"""
===============================================================================
知识图谱构建系统 - 完整版本
===============================================================================
功能：
1. 智能提取DC元数据（无需YAML）
2. 语义分割并标注section_role
3. 为关系添加WHU_HASORIGINALTEXT和llm_weight
4. 为所有节点和关系添加from_section属性
5. 批量注入元数据到Neo4j


===============================================================================
"""

# ==================== 导入依赖 ====================
from typing import List, Optional, Callable, Dict, Any
from llama_index.core import Document
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.core.schema import BaseNode, TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.llms.deepseek import DeepSeek
import os
import re
import nest_asyncio
from datetime import datetime
from tqdm import tqdm
import traceback

# 解决Jupyter/异步环境中的事件循环嵌套问题
nest_asyncio.apply()

# ==================== 模块1: 文档解析与元数据提取 ====================

import json

def load_agent_metadata(markdown_path: str) -> dict:
    """
    加载Agent生成的元数据文件
    
    优先级：
    1. JSON文件（Agent生成）
    2. 默认值（如果没有Agent元数据）
    
    Args:
        markdown_path: Markdown文件路径
        
    Returns:
        dict: DC元数据字典
    """
    base = os.path.splitext(markdown_path)[0]
    json_path = f"{base}_metadata.json"
    
    # 尝试读取Agent生成的JSON
    if os.path.exists(json_path):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                metadata = json.load(f)
            
            # 验证必需字段
            required_fields = ['dc_title', 'dc_author', 'dc_publisher']
            if all(field in metadata for field in required_fields):
                return metadata
            else:
                print(f"⚠️ 元数据文件缺少必需字段: {json_path}")
        
        except Exception as e:
            print(f"⚠️ 读取元数据失败 {json_path}: {e}")
    
    # 如果没有Agent元数据，返回默认值
    filename = os.path.basename(markdown_path)
    print(f"⚠️ 使用默认元数据: {filename}")
    
    return {
        'dc_title': filename.replace('.md', ''),
        'dc_author': 'Unknown',
        'dc_publisher': 'Unknown',
        'dc_creator': 'Unknown',
        'dc_type': 'Research Article',
        'dc_language': 'unknown',
        'dcterms_issued': 'Unknown',
        'dcterms_identifier': f'local:{filename}',
        'source': 'default',
        'source_filename': filename,
        'processed_at': datetime.now().isoformat()
    }


def load_markdown_with_agent_metadata(directory_path: str) -> List[Document]:
    """
    加载Markdown文件并使用Agent生成的元数据
    
    工作流程：
    1. 遍历目录中的.md文件
    2. 读取文件内容
    3. 加载对应的Agent元数据（_metadata.json）
    4. 创建Document对象
    
    Args:
        directory_path: Markdown文件所在目录
        
    Returns:
        List[Document]: 文档对象列表
    """
    documents = []
    
    print(f"\n{'='*60}")
    print(f"📂 加载Markdown文件（使用Agent元数据）")
    print(f"{'='*60}\n")
    
    for filename in os.listdir(directory_path):
        if not filename.endswith('.md'):
            continue
        
        file_path = os.path.join(directory_path, filename)
        
        # 读取Markdown内容
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
        except Exception as e:
            print(f"❌ 读取文件失败 {filename}: {e}")
            continue
        
        # 👇 新方法：读取Agent元数据
        dc_metadata = load_agent_metadata(file_path)
        
        # 创建Document对象
        doc = Document(
            text=content,
            metadata={'filename': filename, **dc_metadata}
        )
        documents.append(doc)
        
        # 打印验证信息
        print(f"📄 {filename}")
        print(f"   标题: {dc_metadata.get('dc_title', 'Unknown')[:60]}")
        print(f"   作者: {dc_metadata.get('dc_author', 'Unknown')}")
        print(f"   来源: {dc_metadata.get('source', 'Unknown')}")
        if dc_metadata.get('dc_subject'):
            print(f"   关键词: {dc_metadata['dc_subject']}")
        print()
    
    print(f"✅ 成功加载 {len(documents)} 个文档\n")
    return documents


def parse_markdown_headers(text: str) -> dict:
    """
    解析Markdown标题层级结构
    
    功能：
    - 识别所有Markdown标题（#, ##, ###等）
    - 构建完整的层级路径（如："第一章 > 1.1节 > 1.1.1小节"）
    - 记录每个标题在文本中的字符位置
    
    Args:
        text (str): Markdown格式的文本内容
        
    Returns:
        dict: {字符位置: 标题路径} 的映射
    """
    lines = text.split('\n')
    position_to_header = {}  # 位置→标题路径映射
    header_stack = []         # 维护当前标题层级栈
    char_position = 0         # 当前字符位置
    
    for line in lines:
        if line.strip().startswith('#'):
            # 计算标题级别（#的数量）
            level = len(line) - len(line.lstrip('#'))
            # 提取标题文本
            title = line.lstrip('#').strip()
            
            # 维护标题栈（移除同级或更低级的标题）
            header_stack = [h for h in header_stack if h[0] < level]
            header_stack.append((level, title))
            
            # 构建完整标题路径（用 > 连接）
            header_path = ' > '.join([h[1] for h in header_stack])
            position_to_header[char_position] = header_path
        
        char_position += len(line) + 1
    
    return position_to_header


def split_by_structure(text: str, chunk_size: int = 800) -> List[str]:
    """
    按Markdown结构智能分割文本
    
    分割策略：
    1. 优先在标题处分割（保持章节完整性）
    2. 其次在段落边界分割（保持段落完整性）
    3. 避免创建过小或过大的块
    
    Args:
        text (str): 待分割的Markdown文本
        chunk_size (int): 目标块大小（字符数），默认800
        
    Returns:
        List[str]: 分割后的文本块列表
    """
    lines = text.split('\n')
    chunks = []       # 最终的文本块
    current = []      # 当前正在构建的块
    size = 0          # 当前块大小
    
    for line in lines:
        line_size = len(line) + 1
        
        # 策略1：在标题处分割（当前块>200字符时）
        if line.strip().startswith('#') and current and size > 200:
            chunks.append('\n'.join(current))
            current, size = [line], line_size
        else:
            current.append(line)
            size += line_size
            
            # 策略2：块超过目标大小时，在空行（段落边界）处分割
            if size > chunk_size and line.strip() == '':
                chunks.append('\n'.join(current[:-1]))
                current, size = [], 0
    
    # 添加最后一个块
    if current:
        chunks.append('\n'.join(current))
    
    return chunks


def create_nodes_with_metadata(doc: Document) -> List[TextNode]:
    """
    创建节点并继承DC元数据
    
    功能：
    1. 使用split_by_structure按结构分割文本
    2. 为每个文本块创建TextNode
    3. 将Document的DC元数据传递给每个Node
    4. 添加块级元数据（filename, chunk_id）
    
    Args:
        doc (Document): 包含文本和DC元数据的Document对象
        
    Returns:
        List[TextNode]: 节点列表，每个包含文本和完整元数据
    """
    # 按结构分割文本
    text_chunks = split_by_structure(doc.text, chunk_size=800)
    nodes = []
    
    # 提取文档级DC元数据（将传递给所有节点）
    doc_dc = {
        k: v for k, v in doc.metadata.items() 
        if k.startswith('dc_') or k.startswith('dcterms_') or k in ['source_filename', 'processed_at']
    }
    
    # 为每个chunk创建节点
    for i, chunk in enumerate(text_chunks):
        if chunk.strip():
            node = TextNode(
                text=chunk,
                metadata={
                    'filename': doc.metadata.get('filename', ''),
                    'chunk_id': i,
                    **doc_dc  # 继承DC元数据
                }
            )
            nodes.append(node)
    
    return nodes


def add_header_paths(nodes: List[BaseNode], original_text: str) -> List[BaseNode]:
    """
    为节点添加标题路径（header_path）
    
    功能：
    1. 解析原文中的所有标题
    2. 为每个节点定位其在原文中的位置
    3. 分配最近的上级标题作为header_path
    
    Args:
        nodes (List[BaseNode]): 节点列表
        original_text (str): 原始文档文本
        
    Returns:
        List[BaseNode]: 添加了header_path的节点列表
    """
    # 解析所有标题及其位置
    position_to_header = parse_markdown_headers(original_text)
    
    # 获取默认标题（文档第一个标题）
    default_title = "Unknown"
    for line in original_text.split('\n')[:20]:
        if line.strip().startswith('#'):
            default_title = line.lstrip('#').strip()
            break
    
    # 为每个节点分配header_path
    for node in nodes:
        if isinstance(node, TextNode):
            if node.metadata is None:
                node.metadata = {}
            
            # 简化定位：使用节点文本前50字符匹配
            node_text = node.get_content()
            clean_node = re.sub(r'\s+', ' ', node_text[:50].strip())
            clean_original = re.sub(r'\s+', ' ', original_text)
            pos = clean_original.find(clean_node)
            
            if pos >= 0:
                # 找到最近的标题
                best_header = default_title
                for header_pos, header in sorted(position_to_header.items(), reverse=True):
                    if header_pos <= pos:
                        best_header = header
                        break
                node.metadata['header_path'] = best_header
            else:
                node.metadata['header_path'] = default_title
    
    return nodes


class SafeSemanticSplitter(SemanticSplitterNodeParser):
    """
    语义分割器（带section_role推断）
    
    功能：
    1. 基于语义相似度的智能分割（继承自父类）
    2. 为每个节点推断section_role（新增）
    3. 保留DC元数据在分割过程中不丢失（新增）
    """
    
    def __init__(self, embed_model=None, section_role_inferrer=None, **kwargs):
        """
        初始化语义分割器
        
        Args:
            embed_model: 嵌入模型（默认使用中文BCE模型）
            section_role_inferrer: section_role推断函数
            **kwargs: 其他参数（similarity_threshold, chunk_size等）
        """
        if embed_model is None:
            embed_model = HuggingFaceEmbedding(model_name="maidalun1020/bce-embedding-base_v1")
        
        super().__init__(embed_model=embed_model, **kwargs)
        
        # 保存section_role推断器
        object.__setattr__(self, '_section_role_inferrer', section_role_inferrer)
    
    def get_nodes_from_documents(self, documents: List[Document], **kwargs) -> List[BaseNode]:
        """
        从文档生成节点，执行语义分割并推断section_role
        
        处理流程：
        1. 调用父类方法执行语义分割
        2. 为每个节点推断section_role
        3. 确保元数据保留
        
        Args:
            documents: 文档列表
            
        Returns:
            List[BaseNode]: 增强后的节点列表
        """
        # 步骤1：执行语义分割
        nodes = super().get_nodes_from_documents(documents, **kwargs)
        
        # 步骤2：获取section_role推断器
        inferrer = getattr(self, '_section_role_inferrer', None)
        if not inferrer:
            return nodes
        
        # 步骤3：为每个节点添加section_role
        enhanced = []
        for node in nodes:
            if isinstance(node, TextNode):
                text = node.get_content()
                metadata = node.metadata or {}
                header = metadata.get("header_path", "")
                
                # 只在没有section_role时才推断
                if "section_role" not in metadata:
                    try:
                        # 调用推断器
                        role = inferrer(text, header)
                        metadata["section_role"] = role
                        node.metadata = metadata
                    except Exception as e:
                        print(f"⚠️ section_role推断失败: {e}")
            enhanced.append(node)
        
        return enhanced


def create_section_inferrer(llm):
    """
    创建混合式section_role推断器（规则+LLM）
    
    推断策略：
    1. 优先使用规则匹配（快速，覆盖80%标准标题）
    2. 规则失败时使用LLM推断（准确，处理20%疑难标题）
    
    Args:
        llm: LangChain的ChatOpenAI实例（DeepSeek）
        
    Returns:
        function: 推断函数，接受(text, header_path)返回section_role
    """
    import re
    
    # 统计LLM调用次数（可选，用于监控）
    llm_call_count = {'count': 0}
    
    def infer(text: str, header: str) -> str:
        """
        推断section_role（混合方案）
        
        Args:
            text (str): 节点文本内容（前200字符）
            header (str): 标题路径
            
        Returns:
            str: section_role类型
        """
        # ========== 第一层：规则匹配（快速，免费）==========
        header_lower = header.lower()
        text_lower = text[:200].lower()
        
        # ===== Abstract（摘要）=====
        if any(k in header_lower for k in [
            'abstract', '摘要', '概述', 'summary', '文摘'
        ]):
            return 'Abstract'
        
        # ===== Introduction（引言）=====
        if any(k in header_lower for k in [
            'introduction', '引言', '前言', '绪论', '背景', 
            'background', '研究背景', '概况', 'overview'
        ]) or re.match(r'^1\.', header_lower):  # 数字模式：1.x
            return 'Introduction'
        
        # ===== Methods_Materials（方法与材料）=====
        if any(k in header_lower for k in [
            # 英文关键词
            'method', 'material', 'procedure', 'experiment', 'experimental',
            'sampling', 'analysis', 'measurement', 'instrument', 'equipment',
            'protocol', 'technique', 'preparation',
            # 中文关键词
            '方法', '材料', '实验', '样品', '检测', '质量控制', '质控',
            '设备', '仪器', '采样', '分析方法', '测定', '试剂', 
            '操作', '步骤', '流程', '制备', '处理', '测试'
        ]) or re.match(r'^2\.', header_lower):  # 数字模式：2.x
            return 'Methods_Materials'
        
        # ===== Results（结果）=====
        if any(k in header_lower for k in [
            # 英文关键词
            'result', 'finding', 'data', 'table', 'figure', 'fig',
            'content', 'concentration', 'level', 'distribution', 
            'observation', 'outcome',
            # 中文关键词
            '结果', '数据', '含量', '浓度', '水平', '分布', 
            '检出', '测定结果', '测量', '观察', '表', '图'
        ]) or re.match(r'^3\.(?!\d)', header_lower):  # 数字模式：3.x（但不是3.x.x）
            return 'Results'
        
        # ===== Discussion（讨论）=====
        if any(k in header_lower for k in [
            # 英文关键词
            'discussion', 'interpretation', 'implication', 'analysis',
            'risk', 'exposure', 'assessment', 'evaluation', 'impact',
            'health', 'safety', 'hazard',
            # 中文关键词
            '讨论', '分析', '评价', '风险', '暴露', '影响', 
            '健康', '安全', '危害', '机制', '原因', '比较'
        ]) or re.match(r'^3\.\d+\.', header_lower):  # 数字模式：3.x.x
            return 'Discussion'
        
        # ===== Conclusion（结论）=====
        if any(k in header_lower for k in [
            'conclusion', 'summary', 'closing', 'outlook', 'perspective',
            '结论', '总结', '小结', '展望', '建议', '对策'
        ]) or re.match(r'^[45]\.', header_lower):  # 数字模式：4.x或5.x
            return 'Conclusion'
        
        # ===== References（参考文献）=====
        if any(k in header_lower for k in [
            'reference', 'bibliography', 'citation', 'literature',
            '参考文献', '文献', '引用'
        ]):
            return 'References'
        
        # ===== 文本特征匹配（辅助判断）=====
        # Methods特征
        if any(k in text_lower for k in [
            'we collected', '我们收集', 'we used', '我们使用',
            'sampling was', '采样', 'according to', '根据', 
            'were analyzed', '进行分析', 'measured by', '测定'
        ]):
            return 'Methods_Materials'
        
        # Results特征
        if any(k in text_lower for k in [
            'p<', 'p =', 'p value', 'p<0.', 'p =0.',
            'significant', '显著', 'was found', '发现', 
            'showed', '显示', 'indicated', '表明'
        ]):
            return 'Results'
        
        # ========== 第二层：LLM兜底（慢但准确）==========
        # 只有规则都不匹配时才调用LLM
        
        if llm is None:
            # 如果没有提供LLM，返回Other
            return 'Other'
        
        try:
            llm_call_count['count'] += 1
            print(f"   🤖 LLM推断 ({llm_call_count['count']}): {header[:40]}...")
            
            # 构建prompt
            prompt = f"""请判断以下学术论文章节属于哪个类型。

标题：{header}

内容前200字：
{text[:200]}

类型选项（只能从中选择一个）：
- Abstract: 摘要
- Introduction: 引言/背景
- Methods_Materials: 方法与材料/实验设计
- Results: 结果/数据/表格
- Discussion: 讨论/分析/评价
- Conclusion: 结论/总结
- References: 参考文献
- Other: 其他

要求：
1. 只回答类型的英文名称（如 "Methods_Materials"）
2. 不要添加任何解释或标点符号
3. 如果不确定，回答 "Other"

答案："""
            
            # 👇 关键修改：使用LangChain API
            # 方式1：使用invoke（推荐）
            response = llm.invoke(prompt)
            
            # 提取结果
            # response是AIMessage对象，需要取.content
            section_role = response.content.strip()
            
            # 验证返回值
            valid_sections = [
                'Abstract', 'Introduction', 'Methods_Materials', 
                'Results', 'Discussion', 'Conclusion', 'References', 'Other'
            ]
            
            if section_role in valid_sections:
                print(f"      ✅ LLM判断: {section_role}")
                return section_role
            else:
                # LLM返回了无效值，尝试模糊匹配
                section_lower = section_role.lower()
                for valid in valid_sections:
                    if valid.lower() in section_lower:
                        print(f"      ✅ LLM判断（修正）: {valid}")
                        return valid
                
                # 完全无法识别
                print(f"      ⚠️ LLM返回无效值: {section_role}，使用Other")
                return 'Other'
        
        except Exception as e:
            print(f"      ⚠️ LLM推断失败: {e}")
            import traceback
            traceback.print_exc()
            return 'Other'
    
    return infer


# 模块2: 元数据更新与关系增强 =

In [3]:
# ==================== 模块2: 元数据更新与关系增强 ====================

def update_metadata_batch(neo4j_driver, filename: str, dc_metadata: dict) -> bool:
    """
    批量更新Chunk/Entity/Relation的元数据（增强版 - 含from_section）
    
    功能：
    1. 为Chunk/Entity/Relation添加DC元数据
    2. 为所有节点和关系添加from_section属性（来自section_role）
    
    原理：
    - Chunk: from_section = section_role（如果section_role存在）
    - Entity: from_section = 关联Chunk的from_section
    - Relation: from_section = 关联Chunk的from_section
    """
    try:
        source_doc = filename.replace('.md', '')
        
        params = {
            'filename': filename,
            'source_doc': source_doc,
            'dc_title': dc_metadata.get('dc_title', source_doc),
            'dc_author': dc_metadata.get('dc_author', 'Unknown'),
            'dc_publisher': dc_metadata.get('dc_publisher', 'Unknown'),
            'dc_creator': dc_metadata.get('dc_creator', 'Unknown'),
            'dcterms_issued': dc_metadata.get('dcterms_issued', 'Unknown'),
            'dcterms_identifier': dc_metadata.get('dcterms_identifier', f'local:{source_doc}')
        }
        
        with neo4j_driver.session() as session:
            # ========== 步骤1：更新Chunk节点（添加from_section）==========
            session.run("""
                MATCH (chunk:Chunk)
                WHERE chunk.dc_title IS NULL
                SET chunk.filename = $filename,
                    chunk.source_doc = $source_doc,
                    chunk.dc_title = $dc_title,
                    chunk.dc_author = $dc_author,
                    chunk.dc_publisher = $dc_publisher,
                    chunk.dc_creator = $dc_creator,
                    chunk.dcterms_issued = $dcterms_issued,
                    chunk.dcterms_identifier = $dcterms_identifier,
                    chunk.from_section = coalesce(chunk.section_role, 'Other'),
                    chunk.processed_at = datetime()
            """, **params)
            
            # ========== 步骤2：更新Entity节点（继承from_section）==========
            session.run("""
                MATCH (chunk:Chunk)-[:FROM_CHUNK]-(entity)
                WHERE chunk.filename = $filename
                  AND entity.dc_title IS NULL
                SET entity.dc_title = $dc_title,
                    entity.dc_author = $dc_author,
                    entity.dc_publisher = $dc_publisher,
                    entity.dc_creator = $dc_creator,
                    entity.dcterms_issued = $dcterms_issued,
                    entity.dcterms_identifier = $dcterms_identifier,
                    entity.source_doc = $source_doc,
                    entity.from_section = chunk.from_section
            """, **params)
            
            # ========== 步骤3：更新Relation关系（继承from_section）==========
            session.run("""
                MATCH (chunk:Chunk)-[:FROM_CHUNK]-(n1)-[r]-(n2)
                WHERE chunk.filename = $filename
                  AND type(r) <> 'FROM_CHUNK'
                  AND r.dc_title IS NULL
                SET r.dc_title = $dc_title,
                    r.dc_author = $dc_author,
                    r.dc_publisher = $dc_publisher,
                    r.dc_creator = $dc_creator,
                    r.dcterms_issued = $dcterms_issued,
                    r.dcterms_identifier = $dcterms_identifier,
                    r.source_doc = $source_doc,
                    r.from_section = chunk.from_section
            """, **params)
            
            print(f"✅ 元数据更新完成: {filename}")
            return True
            
    except Exception as e:
        print(f"❌ 元数据更新失败: {e}")
        import traceback
        traceback.print_exc()
        return False


def enhance_relations(neo4j_driver, filename: str, dc_metadata: dict, llm) -> int:
    """
    为关系和节点增强属性（完整版）
    
    功能：
    1. 为Relation添加 WHU_HASORIGINALTEXT, WHU_HASNAME, llm_weight
    2. 为Node添加 llm_weight（如果缺失）
    
    原理：
    - 优先使用LLM在KG抽取时已提取的属性
    - 对于缺失的属性，使用规则或默认值补充
    """
    try:
        with neo4j_driver.session() as session:
            # ========== 步骤1：增强Relation属性 ==========
            result = session.run("""
                MATCH (chunk:Chunk)-[:FROM_CHUNK]-(n1)-[r]-(n2)
                WHERE chunk.filename = $filename
                  AND type(r) <> 'FROM_CHUNK'
                  AND r.WHU_HASORIGINALTEXT IS NULL
                RETURN id(r) as rel_id, 
                       type(r) as rel_type,
                       n1.WHU_HASNAME as source_name,
                       n1.WHU_HASORIGINALTEXT as source_text, 
                       n2.WHU_HASNAME as target_name,
                       n2.WHU_HASORIGINALTEXT as target_text,
                       chunk.text as chunk_text,
                       r.WHU_HASNAME as existing_name,
                       r.llm_weight as existing_weight
                LIMIT 100
            """, filename=filename)
            
            rels = list(result)
            if not rels:
                print(f"   ℹ️  所有关系已有完整属性")
            
            enhanced_rels = 0
            
            for rel in rels:
                try:
                    chunk_text = rel['chunk_text'] or ''
                    rel_type = rel['rel_type']
                    source_name = rel['source_name'] or ''
                    target_name = rel['target_name'] or ''
                    
                    # ===== 提取 WHU_HASORIGINALTEXT =====
                    # 策略：在chunk_text中找到包含两个实体的句子
                    sentences = chunk_text.replace('。', '.').split('.')
                    original_text = ''
                    
                    for sent in sentences:
                        sent_lower = sent.lower()
                        # 检查句子是否同时包含source和target的关键词
                        if (source_name.lower() in sent_lower and 
                            target_name.lower() in sent_lower):
                            original_text = sent.strip()[:300]
                            break
                    
                    # 如果没找到，使用chunk前200字符
                    if not original_text:
                        original_text = chunk_text[:200].strip() + "..."
                    
                    # ===== 生成 WHU_HASNAME =====
                    # 优先使用LLM已提取的
                    if rel['existing_name']:
                        whu_hasname = rel['existing_name']
                    else:
                        # 否则，基于关系类型生成
                        whu_hasname = rel_type.replace('_', ' ').lower()
                    
                    # ===== 评估 llm_weight =====
                    # 优先使用LLM已提取的
                    if rel['existing_weight'] is not None:
                        llm_weight = float(rel['existing_weight'])
                    else:
                        # 否则，基于文本特征评估
                        llm_weight = 0.5  # 默认中等权重
                        
                        text_lower = original_text.lower()
                        
                        # 高权重关键词（强因果、统计显著）
                        if any(k in text_lower for k in [
                            'significant', 'p<', 'p =', 'p<0.', 'p<0.0',
                            '显著', 'cause', 'lead to', 'result in', 
                            '导致', '产生', 'directly', '直接'
                        ]):
                            llm_weight = 0.85
                        
                        # 中等权重关键词（明确关联）
                        elif any(k in text_lower for k in [
                            'associated', 'correlated', 'related', 'linked',
                            '相关', '关联', 'indicate', '表明', 'showed', '显示'
                        ]):
                            llm_weight = 0.7
                        
                        # 低权重关键词（推测、可能）
                        elif any(k in text_lower for k in [
                            'may', 'might', 'could', 'possibly', 'suggest',
                            '可能', '暗示', 'unclear', '不清楚'
                        ]):
                            llm_weight = 0.4
                    
                    # ===== 更新关系属性 =====
                    session.run("""
                        MATCH ()-[r]->() 
                        WHERE id(r) = $rel_id
                        SET r.WHU_HASORIGINALTEXT = $original_text,
                            r.WHU_HASNAME = $whu_hasname,
                            r.llm_weight = $llm_weight,
                            r.dc_identifier = $dc_identifier
                    """, 
                        rel_id=rel['rel_id'],
                        original_text=original_text,
                        whu_hasname=whu_hasname,
                        llm_weight=llm_weight,
                        dc_identifier=dc_metadata.get('dcterms_identifier', '')
                    )
                    
                    enhanced_rels += 1
                    
                except Exception as e:
                    print(f"      ⚠️ 关系增强失败: {e}")
                    continue
            
            if enhanced_rels > 0:
                print(f"   ✅ 增强了 {enhanced_rels} 个关系属性")
            
            # ========== 步骤2：增强Node的llm_weight（如果缺失）==========
            result = session.run("""
                MATCH (chunk:Chunk)-[:FROM_CHUNK]-(node)
                WHERE chunk.filename = $filename
                  AND node.llm_weight IS NULL
                  AND node.WHU_HASNAME IS NOT NULL
                RETURN id(node) as node_id,
                       labels(node)[0] as label,
                       node.WHU_HASNAME as name,
                       node.WHU_HASORIGINALTEXT as original_text
                LIMIT 100
            """, filename=filename)
            
            nodes = list(result)
            enhanced_nodes = 0
            
            for node_rec in nodes:
                try:
                    label = node_rec['label']
                    name = node_rec['name'] or ''
                    original_text = node_rec['original_text'] or ''
                    
                    # 基于节点类型和文本特征评估权重
                    llm_weight = 0.6  # 默认
                    
                    # 核心实体类型（高权重）
                    if label in ['whu_Pollutant', 'mp_Claim', 'whu_DataSet', 
                                'whu_EnvironmentFeature']:
                        llm_weight = 0.85
                    
                    # 重要支持实体
                    elif label in ['whu_Specimen', 'whu_Computational_Experiment',
                                  'whu_Bio_chemical_Experiment']:
                        llm_weight = 0.75
                    
                    # 标准实体
                    elif label in ['whu_Instrument', 'whu_Method', 'mp_Statement']:
                        llm_weight = 0.6
                    
                    # 次要实体
                    else:
                        llm_weight = 0.5
                    
                    # 根据文本特征微调
                    text_lower = (name + ' ' + original_text).lower()
                    
                    # 强调词 → 提升权重
                    if any(k in text_lower for k in [
                        'significant', 'critical', 'primary', 'main', 'key',
                        '主要', '关键', '重要', '核心'
                    ]):
                        llm_weight = min(llm_weight + 0.1, 1.0)
                    
                    # 更新节点权重
                    session.run("""
                        MATCH (node)
                        WHERE id(node) = $node_id
                        SET node.llm_weight = $llm_weight
                    """, 
                        node_id=node_rec['node_id'],
                        llm_weight=llm_weight
                    )
                    
                    enhanced_nodes += 1
                    
                except Exception as e:
                    continue
            
            if enhanced_nodes > 0:
                print(f"   ✅ 增强了 {enhanced_nodes} 个节点权重")
            
            return enhanced_rels + enhanced_nodes
            
    except Exception as e:
        print(f"❌ 属性增强失败: {e}")
        import traceback
        traceback.print_exc()
        return 0

# 模块3: 主流程

In [ ]:
# ==================== 模块3: 主流程（稳定性优化版 v3） ====================
#
# 在 v2 基础上针对 Neo4j 中断问题做了如下优化：
#   [优化1] Driver 加超时/连接池/keep_alive 参数
#   [优化2] 文档级喘息 + 健康检查（让 Neo4j 有 checkpoint 时间，并及早发现连接问题）
#   [优化3] 文档级重试（单文档失败不中断整批）
#   [优化4] 可选关闭 entity_resolution（降低磁盘膨胀）
#   [优化5] 抛出连接异常时主动重连（防止 driver 状态僵死）
# ============================================================================

import asyncio
import json
import time
from typing import Any, Dict, List, Tuple

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from neo4j.exceptions import ServiceUnavailable, SessionExpired, TransientError

# ---------- 模块级常量 ----------
LOCAL_EMBEDDING_PATH = r"C:/model/bce-embedding-base_v1"

NEO4J_URI         = "bolt://localhost:7687"
NEO4J_USERNAME    = "neo4j"
NEO4J_PASSWORD    = "tomis1cat"
NEO4J_DATABASE    = "neo4j"

DEEPSEEK_API_KEY  = "YOUR_DEEPSEEK_API_KEY"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
DEEPSEEK_MODEL    = "deepseek-v4-flash"

# [优化4] 是否启用实体消解 —— 关闭后磁盘膨胀显著减少，但跨文档同名实体不会合并
PERFORM_ENTITY_RESOLUTION = False

# [优化2] 各级喘息时间（秒）
PAUSE_BETWEEN_SCHEMAS  = 1.0   # 每条 schema 抽取后
PAUSE_BETWEEN_DOCS     = 3.0   # 每个文档后

# [优化3] 单文档失败重试次数
DOC_MAX_RETRIES = 2
# ----------------------------------


# ==================== Schema 加载 ====================

def _load_json(path: str) -> Dict[str, Any]:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"❌ 找不到文件: {path}")
        raise
    except json.JSONDecodeError as e:
        print(f"❌ 无法解析 JSON {path}: {e}")
        raise


def load_schema(base_path: str) -> Tuple[List[Dict], List[Dict], List[List]]:
    entities         = _load_json(f"{base_path}\\entity.json").get("entities", [])
    relations        = _load_json(f"{base_path}\\relation.json").get("relations", [])
    potential_schema = _load_json(f"{base_path}\\potential_schema.json").get("potential_schema", [])
    return entities, relations, potential_schema


# ==================== Neo4j 连接 / 健康检查 ====================

def build_neo4j_driver():
    """[优化1] 构建带稳定性参数的 driver"""
    from neo4j import GraphDatabase
    return GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
        max_connection_lifetime=3600,        # 单连接最长 1 小时，到期自动回收
        max_connection_pool_size=20,         # 连接池上限
        connection_acquisition_timeout=120,  # 等池中连接最多 2 分钟
        connection_timeout=30,               # 建立 TCP 连接超时
        keep_alive=True,                     # TCP keep-alive 防中间链路掉包
    )


def neo4j_is_alive(driver) -> bool:
    """[优化2] 轻量级健康检查"""
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            session.run("RETURN 1").consume()
        return True
    except Exception:
        return False


def wait_for_neo4j(driver, max_wait_sec: int = 120) -> bool:
    """[优化5] Neo4j 短暂中断时等待恢复"""
    print(f"⏳ Neo4j 不可达，等待恢复（最多 {max_wait_sec}s）...")
    start = time.time()
    while time.time() - start < max_wait_sec:
        if neo4j_is_alive(driver):
            print(f"✅ Neo4j 已恢复（等待 {int(time.time() - start)}s）")
            return True
        time.sleep(5)
    print(f"❌ Neo4j 在 {max_wait_sec}s 内未恢复，放弃等待")
    return False


# ==================== Section 工具函数 ====================

def canonical_section(name: str) -> str:
    mapping = {
        'abstract': 'Abstract', 'introduction': 'Introduction',
        'methods': 'Methods_Materials', 'materials': 'Methods_Materials',
        'methods_materials': 'Methods_Materials',
        'results': 'Results', 'discussion': 'Discussion',
        'conclusion': 'Conclusion', 'references': 'References',
        'other': 'Other'
    }
    return mapping.get(name.lower().strip(), name)


def schema_allowed_set(sections: List[str]) -> set:
    if not sections or any(s.lower() == 'all' for s in sections):
        return {'__ALL__'}
    return {canonical_section(s) for s in sections}


def join_nodes_text(nodes: List[Any]) -> str:
    return "\n\n".join([n.get_text().strip() for n in nodes if n.get_text().strip()])


# ==================== 文档处理 ====================

async def _process_document_once(doc, splitter, custom_prompt, potential_schema,
                                 entities, relations, llm, neo4j_driver,
                                 embed_model, weight_llm) -> int:
    """单次处理一个文档（无重试逻辑）"""
    filename = doc.metadata.get('filename', 'Unknown')
    dc_metadata = {
        k: v for k, v in doc.metadata.items()
        if k.startswith('dc_') or k.startswith('dcterms_')
    }

    print(f"\n{'=' * 60}")
    print(f"📄 {filename}: {dc_metadata.get('dc_title', '')[:40]}")
    print(f"{'=' * 60}")

    nodes = create_nodes_with_metadata(doc)
    add_header_paths(nodes, doc.text)
    print(f"📦 粗切: {len(nodes)} 节点")

    doc_blocks = [
        Document(text=n.get_content(), metadata=dict(n.metadata or {}))
        for n in nodes
    ]
    final_nodes = splitter.get_nodes_from_documents(doc_blocks)
    print(f"✅ 细切: {len(final_nodes)} 节点")

    for n in final_nodes:
        md = n.metadata or {}
        md['section_role'] = canonical_section(md.get('section_role', 'Other'))
        n.metadata = md

    role_counts = {}
    for n in final_nodes:
        role = n.metadata.get('section_role', 'Unknown')
        role_counts[role] = role_counts.get(role, 0) + 1
    print(f"🔍 section_role 分布: " +
          ", ".join(f"{r}={c}" for r, c in sorted(role_counts.items(), key=lambda x: -x[1])))

    from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
    processed = 0

    for schema in potential_schema:
        try:
            e1, r, e2 = schema[0], schema[1], schema[2]
            sections = schema[3] if len(schema) > 3 else []
            allowed = schema_allowed_set(sections)

            _entities = [e for e in entities if e.get("label") in (e1, e2)]
            _relations = [rel for rel in relations if rel.get("label") == r]
            if not _entities or not _relations:
                continue

            selected = (final_nodes if '__ALL__' in allowed else
                        [n for n in final_nodes
                         if canonical_section((n.metadata or {}).get('section_role')) in allowed])
            if not selected:
                continue

            text = join_nodes_text(selected)
            if not text.strip():
                continue

            kg_builder = SimpleKGPipeline(
                llm=llm,
                driver=neo4j_driver,
                embedder=embed_model,
                entities=_entities,
                relations=_relations,
                text_splitter=None,
                potential_schema=[schema[:3]],
                from_pdf=False,
                perform_entity_resolution=PERFORM_ENTITY_RESOLUTION,  # [优化4]
                prompt_template=custom_prompt,
                neo4j_database=NEO4J_DATABASE,
            )
            await kg_builder.run_async(text=text)
            processed += 1

            # [优化2] schema 间喘息
            await asyncio.sleep(PAUSE_BETWEEN_SCHEMAS)

        except Exception as e:
            print(f"⚠️ Schema {schema[:3]} 失败: {e}")
            continue

    # 阶段5.5：补充 Chunk 元数据
    print(f"🔧 补充 Chunk 元数据...")
    supplement_count = 0
    try:
        with neo4j_driver.session(database=NEO4J_DATABASE) as session:
            for node in final_nodes:
                text_preview = node.get_content()[:50].strip()
                if not text_preview:
                    continue
                result = session.run("""
                    MATCH (c:Chunk)
                    WHERE c.text CONTAINS $text_preview
                      AND (c.header_path IS NULL OR c.section_role IS NULL)
                    SET c.header_path = $header_path,
                        c.section_role = $section_role
                    RETURN count(c) as count
                """,
                                     text_preview=text_preview,
                                     header_path=node.metadata.get('header_path', 'Unknown'),
                                     section_role=node.metadata.get('section_role', 'Other'))
                updated = result.single()['count']
                if updated > 0:
                    supplement_count += updated
        print(f"   ✅ 补充 {supplement_count} 个 Chunk")
    except Exception as e:
        print(f"   ⚠️ 补充失败: {e}")

    # 阶段6：元数据更新
    if update_metadata_batch(neo4j_driver, filename, dc_metadata):
        enhance_relations(neo4j_driver, filename, dc_metadata, weight_llm)

    return processed


async def process_document_with_retry(doc, **kwargs) -> int:
    """
    [优化3] 带重试的文档处理。
    捕获 Neo4j 连接异常，等待恢复后重试；其它异常直接返回 0 跳过。
    """
    neo4j_driver = kwargs['neo4j_driver']
    filename = doc.metadata.get('filename', 'Unknown')

    for attempt in range(1, DOC_MAX_RETRIES + 2):
        try:
            return await _process_document_once(doc=doc, **kwargs)

        except (ServiceUnavailable, SessionExpired, TransientError) as e:
            print(f"⚠️ [{filename}] Neo4j 连接异常 (尝试 {attempt}/{DOC_MAX_RETRIES + 1}): {e}")
            if attempt > DOC_MAX_RETRIES:
                print(f"❌ [{filename}] 重试上限，跳过此文档")
                return 0
            if not wait_for_neo4j(neo4j_driver, max_wait_sec=180):
                print(f"❌ [{filename}] Neo4j 未恢复，放弃此文档")
                return 0

        except Exception as e:
            print(f"❌ [{filename}] 不可恢复错误: {e}")
            import traceback
            traceback.print_exc()
            return 0

    return 0


# ==================== 主流程 ====================

async def build_knowledge_graph(
        directory_path: str = r".\data\markdown\forTest",
        schema_base_path: str = r".\output",
        api_key: str = DEEPSEEK_API_KEY,
        auto_extract_metadata: bool = True
):
    neo4j_driver = None
    try:
        print("🚀 知识图谱构建开始...")
        print("=" * 80)

        if auto_extract_metadata:
            print("\n🤖 步骤0: Agent 自动提取元数据")
            try:
                # 直接调用模块0 已定义的 process_directory（在 notebook 全局命名空间）
                process_directory(directory=directory_path, clear_existing=True)
                print("✅ Agent 完成\n")
            except NameError:
                print("⚠️ process_directory 未定义，请先运行模块0 cell\n")
            except Exception as e:
                print(f"⚠️ Agent 失败: {e}\n")

        print("\n📦 步骤1: 本地初始化组件")
        print("-" * 80)

        neo4j_driver = build_neo4j_driver()
        neo4j_driver.verify_connectivity()
        with neo4j_driver.session(database=NEO4J_DATABASE) as session:
            session.run("RETURN 1").consume()
            session.run("MATCH (n) DETACH DELETE n")
        print(f"   ✅ Neo4j 已连接并清空，目标数据库: '{NEO4J_DATABASE}'")

        from neo4j_graphrag.llm import OpenAILLM
        llm_neo4j = OpenAILLM(
            model_name=DEEPSEEK_MODEL,
            model_params={"max_tokens": 8000, "temperature": 0.1,
                          "top_p": 0.9, "frequency_penalty": 0.1},
            api_key=api_key, base_url=DEEPSEEK_BASE_URL,
        )
        print(f"   ✅ KG-LLM: {DEEPSEEK_MODEL}")

        from langchain_openai import ChatOpenAI
        llm_langchain = ChatOpenAI(
            model=DEEPSEEK_MODEL, api_key=api_key,
            base_url=DEEPSEEK_BASE_URL, temperature=0, max_tokens=1000,
        )
        print(f"   ✅ Section LLM 已初始化")

        from llama_index.llms.deepseek import DeepSeek
        weight_llm = DeepSeek(model=DEEPSEEK_MODEL, api_key=api_key)
        print(f"   ✅ 权重 LLM 已初始化")

        from neo4j_graphrag.embeddings.sentence_transformers import SentenceTransformerEmbeddings
        embed_model = SentenceTransformerEmbeddings(model=LOCAL_EMBEDDING_PATH)
        print(f"   ✅ KG-Embedding 从本地加载: {LOCAL_EMBEDDING_PATH}")
        print(f"   ℹ️  PERFORM_ENTITY_RESOLUTION = {PERFORM_ENTITY_RESOLUTION}")

        print("\n📋 步骤2: 加载 Schema")
        entities, relations, potential_schema = load_schema(schema_base_path)
        print(f"   实体类型: {len(entities)} | 关系类型: {len(relations)} | "
              f"Schema 组合: {len(potential_schema)}")
        if not potential_schema:
            return False

        print("\n🔧 步骤3: 初始化文本处理器")
        embed_for_split = HuggingFaceEmbedding(model_name=LOCAL_EMBEDDING_PATH)
        section_role_inferrer = create_section_inferrer(llm=llm_langchain)
        splitter = SafeSemanticSplitter(
            embed_model=embed_for_split,
            section_role_inferrer=section_role_inferrer,
            similarity_threshold=0.72,
            chunk_size=300,
            window_size=2,
        )
        print("   ✅ 文本处理器初始化完成")

        print("\n📝 步骤4: 加载 Prompt 模板")
        try:
            with open('custom_prompt.md', 'r', encoding='utf-8') as f:
                custom_prompt = f.read()
            print("   ✅ Prompt 模板加载完成")
        except FileNotFoundError:
            print("   ⚠️ custom_prompt.md 未找到，使用默认")
            custom_prompt = "Extract entities and relations from the text."

        print(f"\n📂 步骤5: 加载 Markdown 文档")
        documents = load_markdown_with_agent_metadata(directory_path)
        print(f"✅ 加载 {len(documents)} 个文档")
        if not documents:
            return False

        print(f"\n🔄 步骤6: 开始处理文档")
        print("=" * 80)

        total_processed = 0
        succeeded_docs = 0
        failed_docs = []

        for idx, doc in enumerate(tqdm(documents, desc="Processing"), start=1):
            # [优化2] 每个文档前快速健康检查
            if not neo4j_is_alive(neo4j_driver):
                print(f"⚠️ Neo4j 不可达，进入等待...")
                if not wait_for_neo4j(neo4j_driver, max_wait_sec=180):
                    print(f"❌ Neo4j 长时间不可达，中止剩余 {len(documents) - idx + 1} 个文档")
                    break

            processed = await process_document_with_retry(
                doc=doc,
                splitter=splitter,
                custom_prompt=custom_prompt,
                potential_schema=potential_schema,
                entities=entities,
                relations=relations,
                llm=llm_neo4j,
                neo4j_driver=neo4j_driver,
                embed_model=embed_model,
                weight_llm=weight_llm,
            )

            if processed > 0:
                total_processed += processed
                succeeded_docs += 1
            else:
                failed_docs.append(doc.metadata.get('filename', 'Unknown'))

            # [优化2] 文档间喘息
            if idx < len(documents):
                await asyncio.sleep(PAUSE_BETWEEN_DOCS)

        print("\n" + "=" * 80)
        print("🔍 步骤7: 最终验证")
        try:
            with neo4j_driver.session(database=NEO4J_DATABASE) as session:
                nodes_count = session.run(
                    "MATCH (n) WHERE n.dc_title IS NOT NULL RETURN count(n) as cnt"
                ).single()['cnt']
                rels_count = session.run(
                    "MATCH ()-[r]->() WHERE r.dc_title IS NOT NULL RETURN count(r) as cnt"
                ).single()['cnt']
                print(f"   节点有 DC 元数据: {nodes_count}")
                print(f"   关系有 DC 元数据: {rels_count}")
        except Exception as e:
            print(f"   ⚠️ 验证查询失败（不影响数据）: {e}")

        print(f"\n🎯 构建完成！")
        print(f"   总文档数: {len(documents)}")
        print(f"   成功文档: {succeeded_docs}")
        print(f"   失败文档: {len(failed_docs)}")
        if failed_docs:
            print(f"   失败列表: {failed_docs}")
        print(f"   处理 schema 数: {total_processed}")
        print("=" * 80)

        return succeeded_docs > 0

    except Exception as e:
        print(f"💥 批量处理失败: {e}")
        import traceback
        traceback.print_exc()
        return False

    finally:
        if neo4j_driver is not None:
            try:
                neo4j_driver.close()
                print("🔒 Neo4j 驱动已关闭")
            except Exception:
                pass

def clear_neo4j_database():
    """清空当前 Neo4j 数据库（NEO4J_DATABASE）的所有节点和关系"""
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(
        NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
    )
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            session.run("MATCH (n) DETACH DELETE n").consume()
        print(f"🗑️  数据库 '{NEO4J_DATABASE}' 已清空")
    finally:
        driver.close()

# ==================== 程序入口点 ====================

if __name__ == "__main__":
    # 在构建知识图谱之前，先清空目标数据库
    clear_neo4j_database()

    result = asyncio.run(build_knowledge_graph(
        directory_path=r".\data\markdown\forTest",
        schema_base_path=r".\output",
        api_key=DEEPSEEK_API_KEY,
        auto_extract_metadata=True,
    ))

    print("\n" + "=" * 80)
    print("🎉 知识图谱构建成功！" if result else "❌ 知识图谱构建失败，请检查日志")
    print("=" * 80)
    print("\n" + "=" * 80)
    print("🎉 知识图谱构建成功！" if result else "❌ 知识图谱构建失败，请检查日志")
    print("=" * 80)

In [7]:
import json

path = r".\output\entity.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 找到 ScienceEvidence
sci = next(e for e in data["entities"] if e["label"] == "whu_ScienceEvidence")

# 在 properties 里找出错位的 SupportGraph
misplaced = [p for p in sci["properties"] if p.get("label") == "whu_SupportGraph"]

if misplaced:
    sg = misplaced[0]
    # 从 properties 移除
    sci["properties"] = [p for p in sci["properties"] if p.get("label") != "whu_SupportGraph"]
    # 加到 entities 末尾
    data["entities"].append(sg)
    print(f"✅ 已把 whu_SupportGraph 从 ScienceEvidence.properties 移到 entities")
else:
    print(f"⚠️ ScienceEvidence.properties 里没找到 whu_SupportGraph，可能已经是其它形式的破损")
    # 打印 properties 里所有 key，看看错位的实体长什么样
    for i, p in enumerate(sci["properties"]):
        print(f"  properties[{i}] 的 keys: {list(p.keys())}")

# 写回
with open(path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# 验证
labels = [e["label"] for e in data["entities"]]
print(f"\n实体总数: {len(labels)}")
print(f"包含 whu_ScienceEvidence: {'whu_ScienceEvidence' in labels}")
print(f"包含 whu_SupportGraph:    {'whu_SupportGraph' in labels}")

✅ 已把 whu_SupportGraph 从 ScienceEvidence.properties 移到 entities

实体总数: 26
包含 whu_ScienceEvidence: True
包含 whu_SupportGraph:    True


In [12]:
import json
with open(r".\output\potential_schema.json", "r", encoding="utf-8") as f:
    ps = json.load(f).get("potential_schema", [])

sci_schemas = [s for s in ps if "whu_ScienceEvidence" in s[:3]]
print(f"涉及 whu_ScienceEvidence 的 schema 条数: {len(sci_schemas)}")
for s in sci_schemas:
    print(f"  {s[:3]}")

涉及 whu_ScienceEvidence 的 schema 条数: 5
  ['whu_ScienceEvidence', 'whu_hasPart', 'whu_DataSet']
  ['whu_ScienceEvidence', 'whu_hasPart', 'whu_Method']
  ['whu_SupportGraph', 'whu_hasPart', 'whu_ScienceEvidence']
  ['whu_ScienceEvidence', 'mp_supports', 'whu_SupportGraph']
  ['whu_ScienceEvidence', 'mp_challenges', 'whu_SupportGraph']


In [ ]:
# ==================== 模块3: 主流程（v5: max_docs 控制 + 日志净化） ====================
import asyncio
import json
import logging
import time
from typing import Any, Dict, List, Tuple

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from neo4j.exceptions import ServiceUnavailable, SessionExpired, TransientError

# ---------- 模块级常量 ----------
LOCAL_EMBEDDING_PATH = r"C:/model/bce-embedding-base_v1"

NEO4J_URI         = "bolt://localhost:7687"
NEO4J_USERNAME    = "neo4j"
NEO4J_PASSWORD    = "tomis1cat"
NEO4J_DATABASE    = "neo4j"

DEEPSEEK_API_KEY  = "YOUR_DEEPSEEK_API_KEY"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
DEEPSEEK_MODEL    = "deepseek-chat"

PERFORM_ENTITY_RESOLUTION = False
PAUSE_BETWEEN_SCHEMAS  = 1.0
PAUSE_BETWEEN_DOCS     = 3.0
DOC_MAX_RETRIES = 2
# ----------------------------------


# ==================== 日志净化 ====================
# 屏蔽 neo4j_graphrag 在 LLM 返回非合法 JSON 时打的 WARNING（其内部已容错，
# 跳过该 chunk 不影响其它 chunk）。仅静音 WARNING，保留 ERROR；不改其逻辑。
def _silence_neo4j_graphrag_warnings():
    for logger_name in (
        "neo4j_graphrag",
        "neo4j_graphrag.experimental.components.entity_relation_extractor",
    ):
        logging.getLogger(logger_name).setLevel(logging.ERROR)


_silence_neo4j_graphrag_warnings()


# ==================== Schema 加载 ====================
def _load_json(path: str) -> Dict[str, Any]:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"❌ 找不到文件: {path}")
        raise
    except json.JSONDecodeError as e:
        print(f"❌ 无法解析 JSON {path}: {e}")
        raise


def load_schema(base_path: str) -> Tuple[List[Dict], List[Dict], List[List]]:
    entities         = _load_json(f"{base_path}\\entity.json").get("entities", [])
    relations        = _load_json(f"{base_path}\\relation.json").get("relations", [])
    potential_schema = _load_json(f"{base_path}\\potential_schema.json").get("potential_schema", [])
    return entities, relations, potential_schema


# ==================== Neo4j 连接 / 健康检查 ====================
def build_neo4j_driver():
    from neo4j import GraphDatabase
    return GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
        max_connection_lifetime=3600,
        max_connection_pool_size=20,
        connection_acquisition_timeout=120,
        connection_timeout=30,
        keep_alive=True,
    )


def neo4j_is_alive(driver) -> bool:
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            session.run("RETURN 1").consume()
        return True
    except Exception:
        return False


def wait_for_neo4j(driver, max_wait_sec: int = 120) -> bool:
    print(f"⏳ Neo4j 不可达，等待恢复（最多 {max_wait_sec}s）...")
    start = time.time()
    while time.time() - start < max_wait_sec:
        if neo4j_is_alive(driver):
            print(f"✅ Neo4j 已恢复（等待 {int(time.time() - start)}s）")
            return True
        time.sleep(5)
    print(f"❌ Neo4j 在 {max_wait_sec}s 内未恢复，放弃等待")
    return False


# ==================== Section 工具函数 ====================
def canonical_section(name: str) -> str:
    mapping = {
        'abstract': 'Abstract', 'introduction': 'Introduction',
        'methods': 'Methods_Materials', 'materials': 'Methods_Materials',
        'methods_materials': 'Methods_Materials',
        'results': 'Results', 'discussion': 'Discussion',
        'conclusion': 'Conclusion', 'references': 'References',
        'other': 'Other'
    }
    return mapping.get(name.lower().strip(), name)


def schema_allowed_set(sections: List[str]) -> set:
    if not sections or any(s.lower() == 'all' for s in sections):
        return {'__ALL__'}
    return {canonical_section(s) for s in sections}


def join_nodes_text(nodes: List[Any]) -> str:
    return "\n\n".join([n.get_text().strip() for n in nodes if n.get_text().strip()])


# ==================== 文档处理 ====================
async def _process_document_once(doc, splitter, custom_prompt, potential_schema,
                                 entities, relations, llm, neo4j_driver,
                                 embed_model, weight_llm) -> int:
    filename = doc.metadata.get('filename', 'Unknown')
    dc_metadata = {
        k: v for k, v in doc.metadata.items()
        if k.startswith('dc_') or k.startswith('dcterms_')
    }

    print(f"\n{'=' * 60}")
    print(f"📄 {filename}: {dc_metadata.get('dc_title', '')[:40]}")
    print(f"{'=' * 60}")

    nodes = create_nodes_with_metadata(doc)
    add_header_paths(nodes, doc.text)
    print(f"📦 粗切: {len(nodes)} 节点")

    doc_blocks = [
        Document(text=n.get_content(), metadata=dict(n.metadata or {}))
        for n in nodes
    ]
    final_nodes = splitter.get_nodes_from_documents(doc_blocks)
    print(f"✅ 细切: {len(final_nodes)} 节点")

    for n in final_nodes:
        md = n.metadata or {}
        md['section_role'] = canonical_section(md.get('section_role', 'Other'))
        n.metadata = md

    role_counts = {}
    for n in final_nodes:
        role = n.metadata.get('section_role', 'Unknown')
        role_counts[role] = role_counts.get(role, 0) + 1
    print(f"🔍 section_role 分布: " +
          ", ".join(f"{r}={c}" for r, c in sorted(role_counts.items(), key=lambda x: -x[1])))

    from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
    processed = 0

    for schema in potential_schema:
        try:
            e1, r, e2 = schema[0], schema[1], schema[2]
            sections = schema[3] if len(schema) > 3 else []
            allowed = schema_allowed_set(sections)

            _entities = [e for e in entities if e.get("label") in (e1, e2)]
            _relations = [rel for rel in relations if rel.get("label") == r]
            if not _entities or not _relations:
                continue

            selected = (final_nodes if '__ALL__' in allowed else
                        [n for n in final_nodes
                         if canonical_section((n.metadata or {}).get('section_role')) in allowed])
            if not selected:
                continue

            text = join_nodes_text(selected)
            if not text.strip():
                continue

            kg_builder = SimpleKGPipeline(
                llm=llm,
                driver=neo4j_driver,
                embedder=embed_model,
                entities=_entities,
                relations=_relations,
                text_splitter=None,
                potential_schema=[schema[:3]],
                from_pdf=False,
                perform_entity_resolution=PERFORM_ENTITY_RESOLUTION,
                prompt_template=custom_prompt,
                neo4j_database=NEO4J_DATABASE,
            )
            await kg_builder.run_async(text=text)
            processed += 1

            await asyncio.sleep(PAUSE_BETWEEN_SCHEMAS)

        except Exception as e:
            print(f"⚠️ Schema {schema[:3]} 失败: {e}")
            continue

    # 阶段5.5：补充 Chunk 元数据
    print(f"🔧 补充 Chunk 元数据...")
    supplement_count = 0
    try:
        with neo4j_driver.session(database=NEO4J_DATABASE) as session:
            for node in final_nodes:
                text_preview = node.get_content()[:50].strip()
                if not text_preview:
                    continue
                result = session.run("""
                    MATCH (c:Chunk)
                    WHERE c.text CONTAINS $text_preview
                      AND (c.header_path IS NULL OR c.section_role IS NULL)
                    SET c.header_path = $header_path,
                        c.section_role = $section_role
                    RETURN count(c) as count
                """,
                                     text_preview=text_preview,
                                     header_path=node.metadata.get('header_path', 'Unknown'),
                                     section_role=node.metadata.get('section_role', 'Other'))
                updated = result.single()['count']
                if updated > 0:
                    supplement_count += updated
        print(f"   ✅ 补充 {supplement_count} 个 Chunk")
    except Exception as e:
        print(f"   ⚠️ 补充失败: {e}")

    if update_metadata_batch(neo4j_driver, filename, dc_metadata):
        enhance_relations(neo4j_driver, filename, dc_metadata, weight_llm)

    return processed


async def process_document_with_retry(doc, **kwargs) -> int:
    neo4j_driver = kwargs['neo4j_driver']
    filename = doc.metadata.get('filename', 'Unknown')

    for attempt in range(1, DOC_MAX_RETRIES + 2):
        try:
            return await _process_document_once(doc=doc, **kwargs)
        except (ServiceUnavailable, SessionExpired, TransientError) as e:
            print(f"⚠️ [{filename}] Neo4j 连接异常 (尝试 {attempt}/{DOC_MAX_RETRIES + 1}): {e}")
            if attempt > DOC_MAX_RETRIES:
                print(f"❌ [{filename}] 重试上限，跳过此文档")
                return 0
            if not wait_for_neo4j(neo4j_driver, max_wait_sec=180):
                print(f"❌ [{filename}] Neo4j 未恢复，放弃此文档")
                return 0
        except Exception as e:
            print(f"❌ [{filename}] 不可恢复错误: {e}")
            import traceback
            traceback.print_exc()
            return 0

    return 0


# ==================== 主流程 ====================
async def build_knowledge_graph(
        directory_path: str = r".\data\markdown\forTest",
        schema_base_path: str = r".\output",
        api_key: str = DEEPSEEK_API_KEY,
        auto_extract_metadata: bool = True,
        max_docs="all",                              # "all" 或正整数
):
    neo4j_driver = None
    try:
        print("🚀 知识图谱构建开始...")
        print("=" * 80)

        if auto_extract_metadata:
            print("\n🤖 步骤0: Agent 自动提取元数据")
            try:
                process_directory(directory=directory_path, clear_existing=True)
                print("✅ Agent 完成\n")
            except NameError:
                print("⚠️ process_directory 未定义，请先运行模块0 cell\n")
            except Exception as e:
                print(f"⚠️ Agent 失败: {e}\n")

        print("\n📦 步骤1: 本地初始化组件")
        print("-" * 80)

        neo4j_driver = build_neo4j_driver()
        neo4j_driver.verify_connectivity()
        with neo4j_driver.session(database=NEO4J_DATABASE) as session:
            session.run("RETURN 1").consume()
            session.run("MATCH (n) DETACH DELETE n")
        print(f"   ✅ Neo4j 已连接并清空，目标数据库: '{NEO4J_DATABASE}'")

        from neo4j_graphrag.llm import OpenAILLM
        llm_neo4j = OpenAILLM(
            model_name=DEEPSEEK_MODEL,
            model_params={"max_tokens": 8000, "temperature": 0.1,
                          "top_p": 0.9, "frequency_penalty": 0.1},
            api_key=api_key, base_url=DEEPSEEK_BASE_URL,
        )
        print(f"   ✅ KG-LLM: {DEEPSEEK_MODEL}")

        from langchain_openai import ChatOpenAI
        llm_langchain = ChatOpenAI(
            model=DEEPSEEK_MODEL, api_key=api_key,
            base_url=DEEPSEEK_BASE_URL, temperature=0, max_tokens=1000,
        )
        print(f"   ✅ Section LLM 已初始化")

        from llama_index.llms.deepseek import DeepSeek
        weight_llm = DeepSeek(model=DEEPSEEK_MODEL, api_key=api_key)
        print(f"   ✅ 权重 LLM 已初始化")

        from neo4j_graphrag.embeddings.sentence_transformers import SentenceTransformerEmbeddings
        embed_model = SentenceTransformerEmbeddings(model=LOCAL_EMBEDDING_PATH)
        print(f"   ✅ KG-Embedding 从本地加载: {LOCAL_EMBEDDING_PATH}")
        print(f"   ℹ️  PERFORM_ENTITY_RESOLUTION = {PERFORM_ENTITY_RESOLUTION}")

        print("\n📋 步骤2: 加载 Schema")
        entities, relations, potential_schema = load_schema(schema_base_path)
        print(f"   实体类型: {len(entities)} | 关系类型: {len(relations)} | "
              f"Schema 组合: {len(potential_schema)}")
        if not potential_schema:
            return False

        print("\n🔧 步骤3: 初始化文本处理器")
        embed_for_split = HuggingFaceEmbedding(model_name=LOCAL_EMBEDDING_PATH)
        section_role_inferrer = create_section_inferrer(llm=llm_langchain)
        splitter = SafeSemanticSplitter(
            embed_model=embed_for_split,
            section_role_inferrer=section_role_inferrer,
            similarity_threshold=0.72,
            chunk_size=300,
            window_size=2,
        )
        print("   ✅ 文本处理器初始化完成")

        print("\n📝 步骤4: 加载 Prompt 模板")
        try:
            with open('custom_prompt.md', 'r', encoding='utf-8') as f:
                custom_prompt = f.read()
            print("   ✅ Prompt 模板加载完成")
        except FileNotFoundError:
            print("   ⚠️ custom_prompt.md 未找到，使用默认")
            custom_prompt = "Extract entities and relations from the text."

        print(f"\n📂 步骤5: 加载 Markdown 文档")
        documents = load_markdown_with_agent_metadata(directory_path)
        print(f"✅ 加载 {len(documents)} 个文档")
        if not documents:
            return False

        # ========== 步骤5.5：按 max_docs 截断 ==========
        if isinstance(max_docs, int) and max_docs > 0:
            if max_docs < len(documents):
                print(f"\n✂️  按 max_docs={max_docs} 截断: {len(documents)} → {max_docs} 个文档")
                documents = documents[:max_docs]
            else:
                print(f"   max_docs={max_docs} ≥ 文档总数，处理全部")
        elif str(max_docs).lower() == "all":
            print(f"   max_docs='all'，处理全部 {len(documents)} 个文档")
        else:
            print(f"\n⚠️ max_docs={max_docs!r} 无效，按全部处理")

        print(f"\n🔄 步骤6: 开始处理文档（共 {len(documents)} 篇）")
        print("=" * 80)

        total_processed = 0
        succeeded_docs = 0
        failed_docs = []

        for idx, doc in enumerate(tqdm(documents, desc="Processing"), start=1):
            if not neo4j_is_alive(neo4j_driver):
                print(f"⚠️ Neo4j 不可达，进入等待...")
                if not wait_for_neo4j(neo4j_driver, max_wait_sec=180):
                    print(f"❌ Neo4j 长时间不可达，中止剩余 {len(documents) - idx + 1} 个文档")
                    break

            processed = await process_document_with_retry(
                doc=doc,
                splitter=splitter,
                custom_prompt=custom_prompt,
                potential_schema=potential_schema,
                entities=entities,
                relations=relations,
                llm=llm_neo4j,
                neo4j_driver=neo4j_driver,
                embed_model=embed_model,
                weight_llm=weight_llm,
            )

            if processed > 0:
                total_processed += processed
                succeeded_docs += 1
            else:
                failed_docs.append(doc.metadata.get('filename', 'Unknown'))

            if idx < len(documents):
                await asyncio.sleep(PAUSE_BETWEEN_DOCS)

        if succeeded_docs > 0:
            for _dep in ("assign_subgraph_properties", "verify_subgraph_assignment", "SUBGRAPH_NAMES"):
                if _dep not in globals():
                    raise RuntimeError(
                        "步骤6.5 需要模块4：请先运行「模块4 子图属性标注」代码 Cell，再执行 build_knowledge_graph。"
                    )
            print("\n" + "=" * 80)
            print("📌 步骤6.5: 子图属性标注")
            print("-" * 80)
            entity_labels_for_sg = {e["label"] for e in entities}
            sg_stats = assign_subgraph_properties(neo4j_driver, schema_base_path)
            verify_subgraph_assignment(neo4j_driver, entity_labels_for_sg)
            _assert_labeled_nodes_exist(sg_stats)
            print(f"   已标注节点: {sg_stats['labeled_nodes_total']}")
            for _sg in SUBGRAPH_NAMES:
                print(f"   {_sg}: {sg_stats['nodes_per_subgraph_membership'][_sg]}")
            print("   ✅ 子图属性标注与验收通过")

        print("\n" + "=" * 80)
        print("🔍 步骤7: 最终验证")
        try:
            with neo4j_driver.session(database=NEO4J_DATABASE) as session:
                nodes_count = session.run(
                    "MATCH (n) WHERE n.dc_title IS NOT NULL RETURN count(n) as cnt"
                ).single()['cnt']
                rels_count = session.run(
                    "MATCH ()-[r]->() WHERE r.dc_title IS NOT NULL RETURN count(r) as cnt"
                ).single()['cnt']
                print(f"   节点有 DC 元数据: {nodes_count}")
                print(f"   关系有 DC 元数据: {rels_count}")
        except Exception as e:
            print(f"   ⚠️ 验证查询失败（不影响数据）: {e}")

        print(f"\n🎯 构建完成！")
        print(f"   总文档数: {len(documents)}")
        print(f"   成功文档: {succeeded_docs}")
        print(f"   失败文档: {len(failed_docs)}")
        if failed_docs:
            print(f"   失败列表: {failed_docs}")
        print(f"   处理 schema 数: {total_processed}")
        print("=" * 80)

        return succeeded_docs > 0

    except Exception as e:
        print(f"💥 批量处理失败: {e}")
        import traceback
        traceback.print_exc()
        return False

    finally:
        if neo4j_driver is not None:
            try:
                neo4j_driver.close()
                print("🔒 Neo4j 驱动已关闭")
            except Exception:
                pass


def clear_neo4j_database():
    """清空当前 Neo4j 数据库（NEO4J_DATABASE）的所有节点和关系"""
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            session.run("MATCH (n) DETACH DELETE n").consume()
        print(f"🗑️  数据库 '{NEO4J_DATABASE}' 已清空")
    finally:
        driver.close()


# ==================== 程序入口点 ====================
if __name__ == "__main__":
    # 控制抽取数量：正整数 = 指定篇数，"all" = 全部
    MAX_DOCS = "all"          # ← 改这里：例如 1 / 3 / "all"

    clear_neo4j_database()

    result = asyncio.run(build_knowledge_graph(
        directory_path=r".\data\markdown\forTest",
        schema_base_path=r".\output",
        api_key=DEEPSEEK_API_KEY,
        auto_extract_metadata=True,
        max_docs=MAX_DOCS,
    ))

    print("\n" + "=" * 80)
    print("🎉 知识图谱构建成功！" if result else "❌ 知识图谱构建失败，请检查日志")
    print("=" * 80)

🗑️  数据库 'neo4j' 已清空
🚀 知识图谱构建开始...

🤖 步骤0: Agent 自动提取元数据

🚀 开始处理 10 个文件
🗑️  模式: 清空并重新生成元数据

📄 doc_01_Dietary intake of minerals and trace elements in rice on the Jamaican market.md
------------------------------------------------------------
   🗑️  已清空: JSON, YAML
✅ 读取: doc_01_Dietary intake of minerals and trace elements in rice on the Jamaican market.md
📝 标题: doc_01_Dietary intake of minerals and trace elements in rice...
🤖 Agent开始决策...
   🔧 调用工具: search_crossref
      ✅ CrossRef找到: Dietary intake of minerals and trace elements in r...
      📄 DOI: 10.1016/j.jfca.2012.01.003
   🔧 调用工具: save_metadata
      💾 已保存: doc_01_Dietary intake of minerals and trace elements in rice on the Jamaican market_metadata.json
   ✅ Agent完成决策
✅ 完成

📄 doc_02_Characterization of mercury species in brown and white rice.md
------------------------------------------------------------
✅ 读取: doc_02_Characterization of mercury species in brown and white rice.md
📝 标题: doc_02_Characterization of mercury species in

: 

# 模块4 子图属性标注

为 Neo4j 实体节点写入 `subgraph` / `subgraphs` 属性，依据 [`output/subgraph_mapping.json`](../output/subgraph_mapping.json)。

| 属性 | 规则 |
|------|------|
| `subgraphs` | `List[str]`，取值 `MPU` / `EBM` / `EEM`，**始终写入** |
| `subgraph` | 仅当节点类型只属于 **一个** 子图时写入标量；跨子图节点 **不写入** |

**本 Cell 可独立运行**（不依赖模块3 的 torch/LLM 导入）。KG 已存在时直接 `run_subgraph_assignment()`。

若通过模块3 `build_knowledge_graph` 自动执行步骤 6.5，须在同 Kernel **先运行本 Cell** 以注册函数。

**严格校验**（失败即 `SubgraphMappingError`）：mapping 与 entity.json 完全一致；所有 ontology 实体节点必须有 `subgraphs`。


In [ ]:
# 模块4 子图属性标注（可独立运行）
from __future__ import annotations

import json
from typing import Any, Dict, List, Set, Tuple

from neo4j import GraphDatabase

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "tomis1cat"
NEO4J_DATABASE = "neo4j"

SUBGRAPH_NAMES: tuple[str, ...] = ("MPU", "EBM", "EEM")
EXCLUDED_NODE_LABELS: frozenset[str] = frozenset({
    "Chunk",
    "MetaPath",
})


class SubgraphMappingError(Exception):
    """subgraph_mapping 或 Neo4j 标注校验失败。"""


def _load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_schema(base_path: str) -> Tuple[List[Dict], List[Dict], List[List]]:
    entities = _load_json(f"{base_path}\\entity.json").get("entities", [])
    relations = _load_json(f"{base_path}\\relation.json").get("relations", [])
    potential_schema = _load_json(f"{base_path}\\potential_schema.json").get(
        "potential_schema", []
    )
    if not entities:
        raise SubgraphMappingError(f"entity.json 无 entities: {base_path}")
    return entities, relations, potential_schema


def build_neo4j_driver():
    return GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
        max_connection_lifetime=3600,
        connection_timeout=30,
    )


def neo4j_is_alive(driver) -> bool:
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            session.run("RETURN 1").consume()
        return True
    except Exception:
        return False


def load_subgraph_mapping_file(mapping_path: str) -> Dict[str, Any]:
    data = _load_json(mapping_path)
    mappings = data.get("mappings")
    if not isinstance(mappings, dict):
        raise SubgraphMappingError(f"mappings 缺失或类型错误: {mapping_path}")
    for sg in SUBGRAPH_NAMES:
        if sg not in mappings:
            raise SubgraphMappingError(f"subgraph_mapping 缺少子图: {sg}")
        labels = mappings[sg]
        if not isinstance(labels, list) or not labels:
            raise SubgraphMappingError(f"子图 {sg} 的实体列表为空")
        if len(labels) != len(set(labels)):
            raise SubgraphMappingError(f"子图 {sg} 存在重复 entity label")
    return data


def build_label_to_subgraphs(mappings: Dict[str, List[str]]) -> Dict[str, List[str]]:
    label_to_sgs: Dict[str, List[str]] = {}
    order = {name: i for i, name in enumerate(SUBGRAPH_NAMES)}
    for sg in SUBGRAPH_NAMES:
        for label in mappings[sg]:
            label_to_sgs.setdefault(label, []).append(sg)
    for label, sgs in label_to_sgs.items():
        label_to_sgs[label] = sorted(set(sgs), key=lambda x: order[x])
    return label_to_sgs


def validate_mapping_vs_entity_schema(
    entity_labels: Set[str],
    label_to_sgs: Dict[str, List[str]],
) -> None:
    mapped = set(label_to_sgs)
    missing = entity_labels - mapped
    extra = mapped - entity_labels
    errors: List[str] = []
    if missing:
        errors.append(
            f"entity.json 未在 subgraph_mapping 中定义: {sorted(missing)}"
        )
    if extra:
        errors.append(
            f"subgraph_mapping 含 entity.json 不存在的 label: {sorted(extra)}"
        )
    if errors:
        raise SubgraphMappingError("\n".join(errors))


def _assign_label_batch(session, label: str, subgraphs: List[str]) -> int:
    params: Dict[str, Any] = {"subgraphs": subgraphs}
    if len(subgraphs) == 1:
        cypher = f"""
        MATCH (n:`{label}`)
        WHERE NOT n:Chunk AND NOT n:MetaPath
        SET n.subgraphs = $subgraphs,
            n.subgraph = $subgraph
        RETURN count(n) AS cnt
        """
        params["subgraph"] = subgraphs[0]
    else:
        cypher = f"""
        MATCH (n:`{label}`)
        WHERE NOT n:Chunk AND NOT n:MetaPath
        SET n.subgraphs = $subgraphs
        REMOVE n.subgraph
        RETURN count(n) AS cnt
        """
    row = session.run(cypher, params).single()
    return int(row["cnt"])


def assign_subgraph_properties(
    driver,
    schema_base_path: str = r".\output",
) -> Dict[str, Any]:
    entities, _, _ = load_schema(schema_base_path)
    entity_labels = {e["label"] for e in entities}

    mapping_path = f"{schema_base_path}\\subgraph_mapping.json"
    mapping_data = load_subgraph_mapping_file(mapping_path)
    label_to_sgs = build_label_to_subgraphs(mapping_data["mappings"])
    validate_mapping_vs_entity_schema(entity_labels, label_to_sgs)

    stats: Dict[str, Any] = {
        "entity_labels_total": len(entity_labels),
        "labeled_nodes_total": 0,
        "by_label": {},
        "nodes_per_subgraph_membership": {sg: 0 for sg in SUBGRAPH_NAMES},
    }

    with driver.session(database=NEO4J_DATABASE) as session:
        for label in sorted(label_to_sgs):
            sgs = label_to_sgs[label]
            cnt = _assign_label_batch(session, label, sgs)
            stats["by_label"][label] = {"node_count": cnt, "subgraphs": sgs}
            stats["labeled_nodes_total"] += cnt
            for sg in sgs:
                stats["nodes_per_subgraph_membership"][sg] += cnt

    return stats


def verify_subgraph_assignment(driver, entity_labels: Set[str]) -> None:
    with driver.session(database=NEO4J_DATABASE) as session:
        missing = session.run(
            """
            MATCH (n)
            WHERE any(l IN labels(n) WHERE l IN $entity_labels)
              AND NOT n:Chunk AND NOT n:MetaPath
              AND n.subgraphs IS NULL
            RETURN count(n) AS cnt
            """,
            entity_labels=list(entity_labels),
        ).single()["cnt"]
        if missing:
            sample = session.run(
                """
                MATCH (n)
                WHERE any(l IN labels(n) WHERE l IN $entity_labels)
                  AND NOT n:Chunk AND NOT n:MetaPath
                  AND n.subgraphs IS NULL
                WITH n, [l IN labels(n) WHERE l IN $entity_labels][0] AS label
                RETURN label, count(*) AS cnt
                ORDER BY cnt DESC
                LIMIT 10
                """,
                entity_labels=list(entity_labels),
            ).data()
            raise SubgraphMappingError(
                f"{missing} 个实体节点缺少 subgraphs 属性。样例: {sample}"
            )

        bad_single = session.run(
            """
            MATCH (n)
            WHERE n.subgraphs IS NOT NULL
              AND size(n.subgraphs) = 1
              AND (n.subgraph IS NULL OR n.subgraph <> n.subgraphs[0])
            RETURN count(n) AS cnt
            """
        ).single()["cnt"]
        if bad_single:
            raise SubgraphMappingError(
                f"{bad_single} 个单属节点 subgraph 与 subgraphs[0] 不一致"
            )

        bad_multi = session.run(
            """
            MATCH (n)
            WHERE n.subgraphs IS NOT NULL
              AND size(n.subgraphs) > 1
              AND n.subgraph IS NOT NULL
            RETURN count(n) AS cnt
            """
        ).single()["cnt"]
        if bad_multi:
            raise SubgraphMappingError(
                f"{bad_multi} 个跨子图节点错误地保留了 subgraph 标量"
            )

        invalid_vals = session.run(
            """
            MATCH (n)
            WHERE n.subgraphs IS NOT NULL
            UNWIND n.subgraphs AS sg
            WITH DISTINCT sg
            WHERE NOT sg IN $allowed
            RETURN collect(sg) AS bad
            """,
            allowed=list(SUBGRAPH_NAMES),
        ).single()["bad"]
        if invalid_vals:
            raise SubgraphMappingError(f"subgraphs 含非法值: {invalid_vals}")

        orphan_entity = session.run(
            """
            MATCH (n:__Entity__)
            WHERE NOT any(l IN labels(n) WHERE l IN $entity_labels)
            RETURN count(n) AS cnt
            """,
            entity_labels=list(entity_labels),
        ).single()["cnt"]
        if orphan_entity:
            raise SubgraphMappingError(
                f"{orphan_entity} 个 __Entity__ 节点缺少 ontology label，无法标注子图"
            )


def run_subgraph_assignment(schema_base_path: str = r".\output") -> Dict[str, Any]:
    driver = build_neo4j_driver()
    try:
        if not neo4j_is_alive(driver):
            raise SubgraphMappingError(
                f"Neo4j 不可达: {NEO4J_URI} / db={NEO4J_DATABASE}"
            )

        entities, _, _ = load_schema(schema_base_path)
        entity_labels = {e["label"] for e in entities}

        print("=" * 60)
        print("模块4: 子图属性标注")
        print("=" * 60)

        stats = assign_subgraph_properties(driver, schema_base_path)
        verify_subgraph_assignment(driver, entity_labels)
        _assert_labeled_nodes_exist(stats)

        print(f"  ontology labels: {stats['entity_labels_total']}")
        print(f"  已标注节点总数: {stats['labeled_nodes_total']}")
        for sg in SUBGRAPH_NAMES:
            print(
                f"  子图 {sg} 成员节点（按 label 计数）: "
                f"{stats['nodes_per_subgraph_membership'][sg]}"
            )
        zero_labels = [
            lbl for lbl, info in stats["by_label"].items() if info["node_count"] == 0
        ]
        if zero_labels:
            print(f"  无实例的 label ({len(zero_labels)}): {zero_labels}")
        print("  验收: 通过")
        print("=" * 60)
        return stats
    finally:
        driver.close()


def _assert_labeled_nodes_exist(stats: Dict[str, Any]) -> None:
    if stats["labeled_nodes_total"] <= 0:
        raise SubgraphMappingError(
            "未标注任何实体节点：Neo4j 中不存在 ontology label 匹配的节点，"
            "或 EXCLUDED_NODE_LABELS 过滤过宽。请先确认 KG 已构建。"
        )


print("✅ 模块4 子图属性标注函数已定义（可独立运行）")
print("   执行: run_subgraph_assignment()")


In [11]:
import json
with open(r".\output\entity.json", "r", encoding="utf-8") as f:
       data = json.load(f)
print("✅ JSON 合法")
print(f"实体总数: {len(data['entities'])}")

✅ JSON 合法
实体总数: 26


# 模块5 chunk去重

实现通过在抽取kg后，通过cypeher在合并重复的chunk 节点，同时建立chunk之间的next_chunk关系

明确是否同具有相同metadata chunk完成了合并

In [13]:
# 模块5 chunk去重（可独立运行）
# -*- coding: utf-8 -*-
import hashlib
from typing import Dict, List, Optional, Any
from neo4j import Driver

class ChunkMerger:
    """精准的Chunk节点合并器 - 仅在同一 filename 内合并，不影响其他节点和关系"""
    
    def __init__(self, driver: Driver):
        self.driver = driver
    
    def merge_duplicate_chunks(self) -> Dict[str, int]:
        """合并重复的Chunk节点（仅合并filename相同的重复），保护其他所有数据"""
        stats = {"original_chunks": 0, "duplicate_groups": 0, "merged_chunks": 0, "final_chunks": 0}
        with self.driver.session() as session:
            stats["original_chunks"] = session.run("MATCH (c:Chunk) RETURN count(c) AS cnt").single()["cnt"]
            duplicate_groups = self._find_duplicate_groups(session)
            stats["duplicate_groups"] = len(duplicate_groups)
            if duplicate_groups:
                print(f"🔍 发现 {len(duplicate_groups)} 组重复Chunk（按 filename 分组）")
                for i, group in enumerate(duplicate_groups):
                    merged_count = self._merge_chunk_group(session, group, i+1)
                    stats["merged_chunks"] += merged_count
                self._rebuild_next_chunk_chain(session)  # 按 filename 分组重建链条
            stats["final_chunks"] = session.run("MATCH (c:Chunk) RETURN count(c) AS cnt").single()["cnt"]
        return stats
    
    def _find_duplicate_groups(self, session) -> List[List[Dict[str, Any]]]:
        """找出所有重复的Chunk组（限定：同一 filename 内部；基于 text 或 embedding）"""
        query = """
        MATCH (c:Chunk)
        WHERE c.filename IS NOT NULL AND c.filename <> ''
        WITH c.filename AS filename,
             c.text AS text,
             CASE WHEN c.embedding IS NOT NULL THEN c.embedding ELSE 'no_embedding' END AS embedding,
             collect({ id: elementId(c), index: coalesce(c.index, 999999), filename: c.filename }) AS chunks
        WHERE size(chunks) > 1
        RETURN filename, text, embedding, chunks
        ORDER BY size(chunks) DESC
        """
        result = session.run(query)
        duplicate_groups = []
        for rec in result:
            chunks = rec["chunks"]
            # 防御：再次确保组内 filename 一致
            fns = {x.get("filename") for x in chunks}
            if len(chunks) > 1 and len(fns) == 1 and list(fns)[0] is not None:
                duplicate_groups.append(chunks)
        return duplicate_groups
    
    def _merge_chunk_group(self, session, chunk_group: List[Dict[str, Any]], group_num: int) -> int:
        """合并一组重复的Chunk节点（默认同一 filename 组）"""
        if len(chunk_group) <= 1:
            return 0
        # 再保险：组内 filename 必须一致，不一致直接跳过
        fn_set = {c["filename"] for c in chunk_group}
        if len(fn_set) != 1:
            print(f"⚠️ 组{group_num}: 检测到混合 filename，跳过")
            return 0
        filename = next(iter(fn_set))
        # 按 index 排序，选最小 index 为保留
        chunk_group.sort(key=lambda x: x["index"])
        target_chunk = chunk_group[0]
        source_chunks = chunk_group[1:]
        print(f"   组{group_num} [file={filename}]: 保留index={target_chunk['index']}，合并{len(source_chunks)}个重复")
        for source_chunk in source_chunks:
            self._transfer_chunk_relationships(session, source_chunk["id"], target_chunk["id"])
        return len(source_chunks)
    
    def _transfer_chunk_relationships(self, session, source_id: str, target_id: str):
        """将源Chunk的所有关系转移到目标Chunk，然后删除源Chunk"""
        self._transfer_incoming_relations(session, source_id, target_id)
        self._transfer_outgoing_relations(session, source_id, target_id)
        session.run("MATCH (s:Chunk) WHERE elementId(s)=$sid DETACH DELETE s", sid=source_id)
    
    def _transfer_incoming_relations(self, session, source_id: str, target_id: str):
        incoming = session.run("""
            MATCH (source:Chunk) WHERE elementId(source)=$sid
            MATCH (target:Chunk) WHERE elementId(target)=$tid
            MATCH (other)-[r]->(source)
            WHERE other <> target
            RETURN elementId(other) AS other_id, type(r) AS rel_type, properties(r) AS rel_props
        """, sid=source_id, tid=target_id).data()
        for rel in incoming:
            session.run(f"""
                MATCH (o) WHERE elementId(o)=$oid
                MATCH (t:Chunk) WHERE elementId(t)=$tid
                MERGE (o)-[nr:{rel['rel_type']}]->(t)
                SET nr += $props
            """, oid=rel["other_id"], tid=target_id, props=rel["rel_props"])
        session.run("MATCH (s:Chunk) WHERE elementId(s)=$sid MATCH (o)-[r]->(s) DELETE r", sid=source_id)
    
    def _transfer_outgoing_relations(self, session, source_id: str, target_id: str):
        outgoing = session.run("""
            MATCH (source:Chunk) WHERE elementId(source)=$sid
            MATCH (target:Chunk) WHERE elementId(target)=$tid
            MATCH (source)-[r]->(other)
            WHERE other <> target
            RETURN elementId(other) AS other_id, type(r) AS rel_type, properties(r) AS rel_props
        """, sid=source_id, tid=target_id).data()
        for rel in outgoing:
            session.run(f"""
                MATCH (t:Chunk) WHERE elementId(t)=$tid
                MATCH (o) WHERE elementId(o)=$oid
                MERGE (t)-[nr:{rel['rel_type']}]->(o)
                SET nr += $props
            """, tid=target_id, oid=rel["other_id"], props=rel["rel_props"])
        session.run("MATCH (s:Chunk) WHERE elementId(s)=$sid MATCH (s)-[r]->(o) DELETE r", sid=source_id)
    
    def _rebuild_next_chunk_chain(self, session):
        """按 filename 分组重建 NEXT_CHUNK 链条（仅处理带 index 的 Chunk）"""
        # 删除所有 Chunk→Chunk 的 NEXT_CHUNK
        deleted = session.run("""
            MATCH (c1:Chunk)-[r:NEXT_CHUNK]->(c2:Chunk) 
            DELETE r 
            RETURN count(r) AS deleted
        """).single()["deleted"]
        if deleted > 0:
            print(f"   清理了 {deleted} 个旧的NEXT_CHUNK关系")
        # 按 filename 分组、按 index 排序重建
        created = session.run("""
            MATCH (c:Chunk)
            WHERE c.index IS NOT NULL AND c.filename IS NOT NULL AND c.filename <> ''
            WITH c.filename AS fn, c ORDER BY fn, c.index
            WITH fn, collect(c) AS chunks
            UNWIND range(0, size(chunks)-2) AS i
            WITH chunks[i] AS cur, chunks[i+1] AS nxt
            CREATE (cur)-[:NEXT_CHUNK]->(nxt)
            RETURN count(*) AS created
        """).single()["created"]
        print(f"   创建了 {created} 个新的NEXT_CHUNK关系（按 filename 分组）")
    
    def verify_merge_result(self) -> Dict[str, Any]:
        """验证合并结果 - 仅检查Chunk相关数据（按 filename 检查重复）"""
        with self.driver.session() as session:
            duplicate_check = session.run("""
                MATCH (c:Chunk)
                WHERE c.filename IS NOT NULL AND c.filename <> ''
                WITH c.filename AS fn, c.text AS text, count(*) AS cnt
                WHERE cnt > 1
                RETURN count(*) AS duplicate_groups
            """).single()["duplicate_groups"]
            chunk_stats = session.run("""
                MATCH (c:Chunk)
                WITH count(c) AS total_chunks
                OPTIONAL MATCH (c1:Chunk)-[r:NEXT_CHUNK]->(c2:Chunk)
                WITH total_chunks, count(r) AS next_relations
                OPTIONAL MATCH (c:Chunk) WHERE c.index IS NOT NULL
                RETURN total_chunks, next_relations, count(c) AS indexed_chunks
            """).single()
            chain_integrity = True
            if chunk_stats["indexed_chunks"] > 1:
                # 由于按 filename 分组重建，这里用分文件期望边数的宽松校验
                expected = session.run("""
                    MATCH (c:Chunk)
                    WHERE c.index IS NOT NULL AND c.filename IS NOT NULL AND c.filename <> ''
                    WITH c.filename AS fn, count(c) AS cnt
                    RETURN reduce(s=0, x IN collect(cnt) | s + CASE WHEN x>0 THEN x-1 ELSE 0 END) AS expected
                """).single()["expected"]
                actual = chunk_stats["next_relations"]
                chain_integrity = (actual >= expected)
            total_stats = session.run("""
                MATCH (n) WITH count(n) AS total_nodes
                MATCH ()-[r]->() WITH total_nodes, count(r) AS total_relations
                RETURN total_nodes, total_relations
            """).single()
            return {
                "chunk_count": chunk_stats["total_chunks"],
                "next_chunk_relations": chunk_stats["next_relations"],
                "indexed_chunks": chunk_stats["indexed_chunks"],
                "duplicate_groups_remaining": duplicate_check,
                "chain_complete": chain_integrity,
                "total_nodes": total_stats["total_nodes"],
                "total_relations": total_stats["total_relations"]
            }

# 主要使用函数
async def merge_chunks_after_extraction(neo4j_driver: Driver) -> bool:
    """在论文抽取完成后合并重复Chunk（仅同 filename 内部），不影响其他数据"""
    print("🔧 开始合并重复Chunk节点...")
    merger = ChunkMerger(neo4j_driver)
    try:
        stats = merger.merge_duplicate_chunks()
        print(f"📊 Chunk合并完成:")
        print(f"   原始Chunk: {stats['original_chunks']}")
        print(f"   重复组数: {stats['duplicate_groups']}")
        print(f"   合并节点: {stats['merged_chunks']}")
        print(f"   最终Chunk: {stats['final_chunks']}")
        rate = ((stats['original_chunks'] - stats['final_chunks']) / stats['original_chunks'] * 100) if stats['original_chunks'] else 0.0
        print(f"   压缩率: {rate:.1f}%")
        verification = merger.verify_merge_result()
        print(f"✅ 验证结果:")
        print(f"   剩余重复组: {verification['duplicate_groups_remaining']}")
        print(f"   NEXT_CHUNK链基本完整: {'是' if verification['chain_complete'] else '否'}")
        print(f"   数据库总节点: {verification['total_nodes']}")
        print(f"   数据库总关系: {verification['total_relations']}")
        return verification["chain_complete"] and verification["duplicate_groups_remaining"] == 0
    except Exception as e:
        print(f"❌ Chunk合并失败: {e}")
        import traceback; traceback.print_exc()
        return False

def cleanup_existing_chunk_duplicates(driver: Driver) -> Dict[str, Any]:
    """清理现有数据库中的重复Chunk - 安全模式（仅同 filename 内部合并）"""
    print("🧽 清理现有Chunk重复...")
    merger = ChunkMerger(driver)
    before = merger.verify_merge_result()
    print(f"清理前: {before['chunk_count']} 个Chunk，{before['duplicate_groups_remaining']} 组重复")
    merged = merger.merge_duplicate_chunks()
    after = merger.verify_merge_result()
    result = {
        "before": before, "merge_stats": merged, "after": after,
        "success": after["duplicate_groups_remaining"] == 0 and after["chain_complete"]
    }
    print(f"清理完成: {after['chunk_count']} 个Chunk，链条{'完整' if after['chain_complete'] else '不完整'}")
    return result

if __name__=="__main__":
    # 示例：按你现有环境获取 driver
    import utilities.return_llm_database
    manager = utilities.return_llm_database.DatabaseManager(remotedatebase=False)
    _, _, neo4j_driver = manager.get_components()
    success = cleanup_existing_chunk_duplicates(neo4j_driver)


🧽 清理现有Chunk重复...
清理前: 3810 个Chunk，328 组重复
🔍 发现 328 组重复Chunk（按 filename 分组）
   组1 [file=doc_06_Heavy Metal pullution in China.md]: 保留index=0，合并49个重复
   组2 [file=doc_06_Heavy Metal pullution in China.md]: 保留index=1，合并49个重复
   组3 [file=doc_01_Dietary intake of minerals and trace elements in rice on the Jamaican market.md]: 保留index=0，合并47个重复
   组4 [file=doc_01_Dietary intake of minerals and trace elements in rice on the Jamaican market.md]: 保留index=1，合并47个重复
   组5 [file=doc_01_Dietary intake of minerals and trace elements in rice on the Jamaican market.md]: 保留index=2，合并47个重复
   组6 [file=doc_02_Characterization of mercury species in brown and white rice.md]: 保留index=0，合并47个重复
   组7 [file=doc_02_Characterization of mercury species in brown and white rice.md]: 保留index=1，合并47个重复
   组8 [file=doc_03_Rice consumption contributes to low level methylmercury exposure in southern China.md]: 保留index=0，合并47个重复
   组9 [file=doc_08_Rice straw-derived biochar amendment enabling.md]: 保留index=0，合并47个重复
   组1